<h1>Group by: split-apply-combine</h1>

<p>By “group by” we are referring to a process involving one or more of the following steps:</p>

<ul>
<li><p><strong>Splitting</strong> the data into groups based on some criteria.</p></li>
<li><p><strong>Applying</strong> a function to each group independently.</p></li>
<li><p><strong>Combining</strong> the results into a data structure.</p></li>
</ul>

<p>Out of these, the split step is the most straightforward. In the apply step, we might wish to do one of the following:</p>

<ul>

<li><p><strong>Aggregation</strong>: compute a summary statistic (or statistics) for each group. Some examples:</p>
<blockquote>
<div><ul>
<li><p>Compute group sums or means.</p></li>
<li><p>Compute group sizes / counts.</p></li>
</ul></div>
</blockquote>
</li>

<li><p><strong>Transformation</strong>: perform some group-specific computations and return a like-indexed object. Some examples:</p>
<blockquote>
<div><ul>
<li><p>Standardize data (zscore) within a group.</p></li>
<li><p>Filling NAs within groups with a value derived from each group.</p></li>
</ul></div>
</blockquote>
</li>

<li><p><strong>Filtration</strong>: discard some groups, according to a group-wise computation that evaluates to True or False. Some examples:</p>
<blockquote>
<div><ul>
<li><p>Discard data that belong to groups with only a few members.</p></li>
<li><p>Filter out data based on the group sum or mean.</p></li>
</ul></div>
</blockquote>
</li>

</ul>

<p>Many of these operations are defined on GroupBy objects.
These operations are similar to those of the <a href="https://pandas.pydata.org/docs/user_guide/basics.html#basics-aggregate">aggregating API</a>, <a href="https://pandas.pydata.org/docs/user_guide/window.html#window-overview">window API</a>, and <a href="https://pandas.pydata.org/docs/user_guide/timeseries.html#timeseries-aggregate">resample API</a>.</p>

<p>It is possible that a given operation does not fall into one of these categories or is some combination of them.
In such a case, it may be possible to compute the operation using GroupBy’s <code>apply</code> method.
This method will examine the results of the apply step and try to sensibly combine them into a single result if it doesn’t fit into either of the above three categories.</p>

<div class="alert alert-block alert-info">
<p>Note</p>
<p>An operation that is split into multiple steps using built-in GroupBy operations will be more efficient than using the <code>apply</code> method with a user-defined Python function.</p>
</div>

<p>The name GroupBy should be quite familiar to those who have used a SQL-based tool (or <code>itertools</code>), in which you can write code like:</p>

<pre>
SELECT Column1, Column2, mean(Column3), sum(Column4)
FROM SomeTable
GROUP BY Column1, Column2
</pre>

<p>We aim to make operations like this natural and easy to express using pandas. We’ll address each area of GroupBy functionality, then provide some non-trivial examples / use cases.</p>

<p>See the <a href="https://pandas.pydata.org/docs/user_guide/cookbook.html#cookbook-grouping">cookbook</a> for some advanced strategies.</p>

# <h2>Splitting an object into groups</h2>

<p>The abstract definition of grouping is to provide a mapping of labels to group names. To create a GroupBy object (more on what the GroupBy object is later), you may do the following:</p>

In [4]:
import pandas as pd
import numpy as np

In [5]:
speeds = pd.DataFrame(
    [
        ("bird", "Falconiformes", 389.0),
        ("bird", "Psittaciformes", 24.0),
        ("mammal", "Carnivora", 80.2),
        ("mammal", "Primates", np.nan),
        ("mammal", "Carnivora", 58),
    ],
    index=["falcon", "parrot", "lion", "monkey", "leopard"],
    columns=("class", "order", "max_speed"),
)

In [6]:
speeds

,class,order,max_speed
falcon,bird,Falconiformes,389.0
parrot,bird,Psittaciformes,24.0
lion,mammal,Carnivora,80.2
monkey,mammal,Primates,NaN
leopard,mammal,Carnivora,58.0


In [7]:
grouped = speeds.groupby("class")

In [8]:
grouped = speeds.groupby(["class", "order"])

<p>The mapping can be specified many different ways:</p>
<ul>
<li><p>A Python function, to be called on each of the index labels.</p></li>
<li><p>A list or NumPy array of the same length as the index.</p></li>
<li><p>A dict or <code>Series</code>, providing a <code>label -&gt; group name</code> mapping.</p></li>
<li><p>For <code>DataFrame</code> objects, a string indicating either a column name or
an index level name to be used to group.</p></li>
<li><p>A list of any of the above things.</p></li>
</ul>

<p>Collectively we refer to the grouping objects as the <strong>keys</strong>. For example,
consider the following <code>DataFrame</code>:</p>

<div class="alert alert-block alert-info">
<p>Note</p>
<p>A string passed to <code>groupby</code> may refer to either a column or an index level.
If a string matches both a column name and an index level name, a
<code>ValueError</code> will be raised.</p>

In [9]:
df = pd.DataFrame(
    {
        "A": ["foo", "bar", "foo", "bar", "foo", "bar", "foo", "foo"],
        "B": ["one", "one", "two", "three", "two", "two", "one", "three"],
        "C": np.random.randn(8),
        "D": np.random.randn(8),
    }
)

In [10]:
df

,A,B,C,D
0,foo,one,-1.209501,0.995005
1,bar,one,-0.608950,0.686688
2,foo,two,-0.493289,0.727253
3,bar,three,-0.388262,0.513634
4,foo,two,-0.899211,-0.440537
5,bar,two,0.587808,0.970356
6,foo,one,1.321277,1.593140
7,foo,three,0.261124,1.070727


<p>On a DataFrame, we obtain a GroupBy object by calling <a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html#pandas.DataFrame.groupby" title="pandas.DataFrame.groupby"><code>groupby()</code></a>.
This method returns a <code>pandas.api.typing.DataFrameGroupBy</code> instance.
We could naturally group by either the <code>A</code> or <code>B</code> columns, or both:</p>

In [11]:
grouped = df.groupby("A")

In [12]:
grouped = df.groupby("B")

In [13]:
grouped = df.groupby(["A", "B"])

<div class="alert alert-block alert-info">
<p>Note</p>
<p><code>df.groupby('A')</code> is just syntactic sugar for <code>df.groupby(df['A'])</code>.</p>
</div>

<p>If we also have a MultiIndex on columns <code>A</code> and <code>B</code>, we can group by all the columns except the one we specify:</p>

In [14]:
df2 = df.set_index(["A", "B"])

In [15]:
grouped = df2.groupby(level=df2.index.names.difference(["B"]))

In [16]:
grouped.sum()

,C,D
A,,
bar,-0.409404,2.170678
foo,-1.019600,3.945588


<p>The above GroupBy will split the DataFrame on its index (rows). To split by columns, first do
a transpose:</p>

In [17]:
def get_letter_type(letter):
    if letter.lower() in 'aeiou':
        return 'vowel'
    else:
        return 'consonant'

In [18]:
grouped = df.T.groupby(get_letter_type)

<p>pandas <a href="https://pandas.pydata.org/docs/reference/api/pandas.Index.html#pandas.Index"><code>Index</code></a> objects support duplicate values.
If a non-unique index is used as the group key in a groupby operation, all values for the same index value will be considered to be in one group and thus the output of aggregation functions will only contain unique index values:</p>

In [19]:
index = [1, 2, 3, 1, 2, 3]

In [20]:
s = pd.Series([1, 2, 3, 10, 20, 30], index=index)

In [21]:
s

1     1
2     2
3     3
1    10
2    20
3    30
dtype: int64

In [22]:
grouped = s.groupby(level=0)

In [23]:
grouped.first()

1    1
2    2
3    3
dtype: int64

In [24]:
grouped.last()

1    10
2    20
3    30
dtype: int64

In [25]:
grouped.sum()

1    11
2    22
3    33
dtype: int64

<p>Note that <strong>no splitting occurs</strong> until it’s needed. Creating the GroupBy object only verifies that you’ve passed a valid mapping.</p>

<div class="alert alert-block alert-info">
<p>Note</p>
<p>Many kinds of complicated data manipulations can be expressed in terms of GroupBy operations (though it can’t be guaranteed to be the most efficient implementation). You can get quite creative with the label mapping functions.</p>
</div>

## <h3>GroupBy sorting</h3>

<p>By default the group keys are sorted during the <code>groupby</code> operation. You may however pass <code>sort=False</code> for potential speedups. With <code>sort=False</code> the order among group-keys follows the order of appearance of the keys in the original dataframe:</p>

In [26]:
df2 = pd.DataFrame({"X": ["B", "B", "A", "A"], "Y": [1, 2, 3, 4]})

In [27]:
df2.groupby(["X"]).sum()

,Y
X,
A,7
B,3


In [28]:
df2.groupby(["X"], sort=False).sum()

,Y
X,
B,3
A,7


<p>Note that <code>groupby</code> will preserve the order in which <em>observations</em> are sorted <em>within</em> each group.
For example, the groups created by <code>groupby()</code> below are in the order they appeared in the original <code>DataFrame</code>:</p>

In [29]:
df3 = pd.DataFrame({"X": ["A", "B", "A", "B"], "Y": [1, 4, 3, 2]})

In [30]:
df3.groupby("X").get_group("A")

,X,Y
0,A,1
2,A,3


In [31]:
df3.groupby(["X"]).get_group("B")

/var/folders/rt/k30tjkvx04vdq1qxvtwsb47m0000gn/T/ipykernel_85300/862706516.py:1: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.
  df3.groupby(["X"]).get_group("B")


,X,Y
1,B,4
3,B,2


### <h4>GroupBy dropna</h4>

<p>By default <code>NA</code> values are excluded from group keys during the <code>groupby</code> operation.
However, in case you want to include <code>NA</code> values in group keys, you could pass <code>dropna=False</code> to achieve it.</p>

In [32]:
df_list = [[1, 2, 3], [1, None, 4], [2, 1, 3], [1, 2, 2]]

In [33]:
df_dropna = pd.DataFrame(df_list, columns=["a", "b", "c"])

In [34]:
df_dropna

,a,b,c
0,1,2.0,3
1,1,NaN,4
2,2,1.0,3
3,1,2.0,2


In [35]:
# Default ``dropna`` is set to True, which will exclude NaNs in keys
df_dropna.groupby(by=["b"], dropna=True).sum()

,a,c
b,,
1.0,2,3
2.0,2,5


In [36]:
# In order to allow NaN in keys, set dropna to False
df_dropna.groupby(by=["b"], dropna=False).sum()

,a,c
b,,
1.0,2,3
2.0,2,5
NaN,1,4


<p>The default setting of <code>dropna</code> argument is <code>True</code> which means <code>NA</code> are not included in group keys.</p>

## <h3>GroupBy object attributes</h3>

<p>The <code>groups</code> attribute is a dictionary whose keys are the computed unique groups and corresponding values are the axis labels belonging to each group. In the above example we have:</p>

In [37]:
df.groupby("A").groups

{'bar': [1, 3, 5], 'foo': [0, 2, 4, 6, 7]}

In [38]:
df.T.groupby(get_letter_type).groups

{'consonant': ['B', 'C', 'D'], 'vowel': ['A']}

<p>Calling the standard Python <code>len</code> function on the GroupBy object returns the number of groups, which is the same as the length of the <code>groups</code> dictionary:</p>

In [39]:
grouped = df.groupby(["A", "B"])

In [40]:
grouped.groups

{('bar', 'one'): [1], ('bar', 'three'): [3], ('bar', 'two'): [5], ('foo', 'one'): [0, 6], ('foo', 'three'): [7], ('foo', 'two'): [2, 4]}

In [41]:
len(grouped)

6

<p><code>GroupBy</code> will tab complete column names, GroupBy operations, and other attributes:</p>

In [42]:
n = 10

In [43]:
weight = np.random.normal(166, 20, size=n)

In [44]:
height = np.random.normal(60, 10, size=n)

In [45]:
time = pd.date_range("1/1/2000", periods=n)

In [46]:
gender = np.random.choice(["male", "female"], size=n)

In [47]:
df = pd.DataFrame(
    {"height": height, "weight": weight, "gender": gender}, index=time
)

In [48]:
df

,height,weight,gender
2000-01-01,60.460520,179.387254,male
2000-01-02,62.974591,133.779025,female
2000-01-03,52.583023,178.676835,male
2000-01-04,44.677055,153.513279,female
2000-01-05,67.845671,159.985484,male
2000-01-06,71.553626,115.923960,male
2000-01-07,61.085390,183.130984,female
2000-01-08,59.169928,169.390568,female
2000-01-09,66.738804,161.107766,female
2000-01-10,45.844737,187.256352,male


In [49]:
gb = df.groupby("gender")

<pre>
In [46]: gb.<TAB>  # noqa: E225, E999
gb.agg        gb.boxplot    gb.cummin     gb.describe   gb.filter     gb.get_group  gb.height     gb.last       gb.median     gb.ngroups    gb.plot       gb.rank       gb.std        gb.transform
gb.aggregate  gb.count      gb.cumprod    gb.dtype      gb.first      gb.groups     gb.hist       gb.max        gb.min        gb.nth        gb.prod       gb.resample   gb.sum        gb.var
gb.apply      gb.cummax     gb.cumsum     gb.fillna     gb.gender     gb.head       gb.indices    gb.mean       gb.name       gb.ohlc       gb.quantile   gb.size       gb.tail       gb.weight
</pre>

## <h3>GroupBy with MultiIndex</h3>

<p>With <a href="https://pandas.pydata.org/docs/user_guide/advanced.html#advanced-hierarchical">hierarchically-indexed data</a>, it’s quite
natural to group by one of the levels of the hierarchy.</p>

<p>Let’s create a Series with a two-level <code>MultiIndex</code>.</p>

In [50]:
arrays = [
    ["bar", "bar", "baz", "baz", "foo", "foo", "qux", "qux"],
    ["one", "two", "one", "two", "one", "two", "one", "two"],
]

In [51]:
index = pd.MultiIndex.from_arrays(arrays, names=["first", "second"])

In [52]:
s = pd.Series(np.random.randn(8), index=index)

In [53]:
s

first  second
bar    one       0.252282
       two       1.540140
baz    one       0.964653
       two      -0.266567
foo    one       0.927404
       two      -0.371515
qux    one      -2.094662
       two      -0.375765
dtype: float64

<p>We can then group by one of the levels in <code>s</code>.</p>

In [54]:
grouped = s.groupby(level=0)

In [55]:
grouped.sum()

first
bar    1.792422
baz    0.698086
foo    0.555889
qux   -2.470427
dtype: float64

<p>If the MultiIndex has names specified, these can be passed instead of the level number:</p>

In [56]:
s.groupby(level="second").sum()

second
one    0.049677
two    0.526293
dtype: float64

<p>Grouping with multiple levels is supported.</p>

In [57]:
arrays = [
    ["bar", "bar", "baz", "baz", "foo", "foo", "qux", "qux"],
    ["doo", "doo", "bee", "bee", "bop", "bop", "bop", "bop"],
    ["one", "two", "one", "two", "one", "two", "one", "two"],
]

In [58]:
index = pd.MultiIndex.from_arrays(arrays, names=["first", "second", "third"])

In [59]:
s = pd.Series(np.random.rand(8), index=index)

In [60]:
s

first  second  third
bar    doo     one      0.508752
               two      0.714705
baz    bee     one      0.968208
               two      0.874563
foo    bop     one      0.344287
               two      0.087067
qux    bop     one      0.868266
               two      0.004492
dtype: float64

In [61]:
s.groupby(level=["first", "second"]).sum()

first  second
bar    doo       1.223457
baz    bee       1.842771
foo    bop       0.431354
qux    bop       0.872758
dtype: float64

<p>Index level names may be supplied as keys.</p>

In [62]:
s.groupby(["first", "second"]).sum()

first  second
bar    doo       1.223457
baz    bee       1.842771
foo    bop       0.431354
qux    bop       0.872758
dtype: float64

<p>More on the <code>sum</code> function and aggregation later.</p>

## <h3>Grouping DataFrame with Index levels and columns</h3>

<p>A DataFrame may be grouped by a combination of columns and index levels. You can specify both column and index names, or use a <a href="https://pandas.pydata.org/docs/reference/api/pandas.Grouper.html#pandas.Grouper" title="pandas.Grouper"><code>Grouper</code></a>.</p>

<p>Let’s first create a DataFrame with a MultiIndex:</p>

In [63]:
arrays = [
    ["bar", "bar", "baz", "baz", "foo", "foo", "qux", "qux"],
    ["one", "two", "one", "two", "one", "two", "one", "two"],
]

In [64]:
index = pd.MultiIndex.from_arrays(arrays, names=["first", "second"])

In [65]:
df = pd.DataFrame({"A": [1, 1, 1, 1, 2, 2, 3, 3], "B": np.arange(8)}, index=index)

In [66]:
df

A  B
first second      
bar   one     1  0
      two     1  1
baz   one     1  2
      two     1  3
foo   one     2  4
      two     2  5
qux   one     3  6
      two     3  7

<p>Then we group <code>df</code> by the <code>second</code> index level and the <code>A</code> column.</p>

In [67]:
df.groupby([pd.Grouper(level=1), "A"]).sum()

B
second A   
one    1  2
       2  4
       3  6
two    1  4
       2  5
       3  7

<p>Index levels may also be specified by name.</p>

In [68]:
df.groupby([pd.Grouper(level="second"), "A"]).sum()

B
second A   
one    1  2
       2  4
       3  6
two    1  4
       2  5
       3  7

<p>Index level names may be specified as keys directly to <code>groupby</code>.</p>

In [69]:
df.groupby(["second", "A"]).sum()

B
second A   
one    1  2
       2  4
       3  6
two    1  4
       2  5
       3  7

## <h3>DataFrame column selection in GroupBy</h3>

<p>Once you have created the GroupBy object from a DataFrame, you might want to do something different for each of the columns.
Thus, by using <code>[]</code> on the GroupBy object in a similar way as the one used to get a column from a DataFrame, you can do:</p>

In [70]:
df = pd.DataFrame(
    {
        "A": ["foo", "bar", "foo", "bar", "foo", "bar", "foo", "foo"],
        "B": ["one", "one", "two", "three", "two", "two", "one", "three"],
        "C": np.random.randn(8),
        "D": np.random.randn(8),
    }
)

In [71]:
df

,A,B,C,D
0,foo,one,0.508658,0.429526
1,bar,one,-0.539168,-0.465201
2,foo,two,-0.288091,0.615694
3,bar,three,0.569972,0.725827
4,foo,two,-1.724389,-0.866913
5,bar,two,-1.498925,0.274429
6,foo,one,0.905717,-1.304709
7,foo,three,0.173850,0.056050


In [72]:
grouped = df.groupby(["A"])

In [73]:
grouped_C = grouped["C"]

In [74]:
grouped_D = grouped["D"]

<p>This is mainly syntactic sugar for the alternative, which is much more verbose:</p>

In [75]:
df["C"].groupby(df["A"])

<p>Additionally, this method avoids recomputing the internal grouping information derived from the passed key.</p>

<p>You can also include the grouping columns if you want to operate on them.</p>

In [76]:
grouped[["A", "B"]].sum()

,A,B
A,,
bar,barbarbar,onethreetwo
foo,foofoofoofoofoo,onetwotwoonethree


# <h2>Iterating through groups</h2>

<p>With the GroupBy object in hand, iterating through the grouped data is very natural and functions similarly to <a href="https://docs.python.org/3/library/itertools.html#itertools.groupby" title="(in Python v3.13)"><code >itertools.groupby()</code></a>:</p>

In [77]:
grouped = df.groupby('A')

In [78]:
for name, group in grouped:
    print(name)
    print(group)

bar
     A      B         C         D
1  bar    one -0.539168 -0.465201
3  bar  three  0.569972  0.725827
5  bar    two -1.498925  0.274429
foo
     A      B         C         D
0  foo    one  0.508658  0.429526
2  foo    two -0.288091  0.615694
4  foo    two -1.724389 -0.866913
6  foo    one  0.905717 -1.304709
7  foo  three  0.173850  0.056050


<p>In the case of grouping by multiple keys, the group name will be a tuple:</p>

In [79]:
for name, group in df.groupby(['A', 'B']):
    print(name)
    print(group)

('bar', 'one')
     A    B         C         D
1  bar  one -0.539168 -0.465201
('bar', 'three')
     A      B         C         D
3  bar  three  0.569972  0.725827
('bar', 'two')
     A    B         C         D
5  bar  two -1.498925  0.274429
('foo', 'one')
     A    B         C         D
0  foo  one  0.508658  0.429526
6  foo  one  0.905717 -1.304709
('foo', 'three')
     A      B        C        D
7  foo  three  0.17385  0.05605
('foo', 'two')
     A    B         C         D
2  foo  two -0.288091  0.615694
4  foo  two -1.724389 -0.866913


<p>See <a href="https://pandas.pydata.org/docs/user_guide/timeseries.html#timeseries-iterating-label">Iterating through groups</a>.</p>

# <h2>Selecting a group</h2>

<p>A single group can be selected using <a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.get_group.html" title="pandas.core.groupby.DataFrameGroupBy.get_group"><code>DataFrameGroupBy.get_group()</code></a>:</p>

In [80]:
grouped.get_group("bar")

,A,B,C,D
1,bar,one,-0.539168,-0.465201
3,bar,three,0.569972,0.725827
5,bar,two,-1.498925,0.274429


<p>Or for an object grouped on multiple columns:</p>

In [81]:
df.groupby(["A", "B"]).get_group(("bar", "one"))

,A,B,C,D
1,bar,one,-0.539168,-0.465201


# <h2>Aggregation</h2>

<p>An aggregation is a GroupBy operation that reduces the dimension of the grouping object. The result of an aggregation is, or at least is treated as, a scalar value for each column in a group. For example, producing the sum of each column in a group of values.</p>

In [82]:
animals = pd.DataFrame(
    {
        "kind": ["cat", "dog", "cat", "dog"],
        "height": [9.1, 6.0, 9.5, 34.0],
        "weight": [7.9, 7.5, 9.9, 198.0],
    }
)

In [83]:
animals

,kind,height,weight
0,cat,9.1,7.9
1,dog,6.0,7.5
2,cat,9.5,9.9
3,dog,34.0,198.0


In [84]:
animals.groupby("kind").sum()

,height,weight
kind,,
cat,18.6,17.8
dog,40.0,205.5


<p>In the result, the keys of the groups appear in the index by default. They can be instead included in the columns by passing <code>as_index=False</code>.</p>

In [85]:
animals.groupby("kind", as_index=False).sum()

,kind,height,weight
0,cat,18.6,17.8
1,dog,40.0,205.5


## <h3>Built-in aggregation methods</h3>

<p>Many common aggregations are built-in to GroupBy objects as methods. Of the methods listed below, those with a <code>*</code> do <em>not</em> have an efficient, GroupBy-specific, implementation.</p>

<table class="table">
<colgroup>
<col style="width: 20.0%">
<col style="width: 80.0%">
</colgroup>
<thead>
<tr class="row-odd"><th class="head"><p>Method</p></th>
<th class="head"><p>Description</p></th>
</tr>
</thead>

<tbody>
<tr class="row-even"><td><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.any.html" title="pandas.core.groupby.DataFrameGroupBy.any"><code>any()</code></a></p></td>
<td><p>Compute whether any of the values in the groups are truthy</p></td>
</tr>
<tr class="row-odd"><td><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.all.html" title="pandas.core.groupby.DataFrameGroupBy.all"><code>all()</code></a></p></td>
<td><p>Compute whether all of the values in the groups are truthy</p></td>
</tr>
<tr class="row-even"><td><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.count.html" title="pandas.core.groupby.DataFrameGroupBy.count"><code>count()</code></a></p></td>
<td><p>Compute the number of non-NA values in the groups</p></td>
</tr>
<tr class="row-odd"><td><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.cov.html" title="pandas.core.groupby.DataFrameGroupBy.cov"><code>cov()</code></a> *</p></td>
<td><p>Compute the covariance of the groups</p></td>
</tr>
<tr class="row-even"><td><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.first.html" title="pandas.core.groupby.DataFrameGroupBy.first"><code>first()</code></a></p></td>
<td><p>Compute the first occurring value in each group</p></td>
</tr>
<tr class="row-odd"><td><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.idxmax.html" title="pandas.core.groupby.DataFrameGroupBy.idxmax"><code>idxmax()</code></a></p></td>
<td><p>Compute the index of the maximum value in each group</p></td>
</tr>
<tr class="row-even"><td><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.idxmin.html" title="pandas.core.groupby.DataFrameGroupBy.idxmin"><code>idxmin()</code></a></p></td>
<td><p>Compute the index of the minimum value in each group</p></td>
</tr>
<tr class="row-odd"><td><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.last.html" title="pandas.core.groupby.DataFrameGroupBy.last"><code>last()</code></a></p></td>
<td><p>Compute the last occurring value in each group</p></td>
</tr>
<tr class="row-even"><td><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.max.html" title="pandas.core.groupby.DataFrameGroupBy.max"><code>max()</code></a></p></td>
<td><p>Compute the maximum value in each group</p></td>
</tr>
<tr class="row-odd"><td><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.mean.html" title="pandas.core.groupby.DataFrameGroupBy.mean"><code>mean()</code></a></p></td>
<td><p>Compute the mean of each group</p></td>
</tr>
<tr class="row-even"><td><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.median.html" title="pandas.core.groupby.DataFrameGroupBy.median"><code>median()</code></a></p></td>
<td><p>Compute the median of each group</p></td>
</tr>
<tr class="row-odd"><td><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.min.html" title="pandas.core.groupby.DataFrameGroupBy.min"><code>min()</code></a></p></td>
<td><p>Compute the minimum value in each group</p></td>
</tr>
<tr class="row-even"><td><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.nunique.html" title="pandas.core.groupby.DataFrameGroupBy.nunique"><code>nunique()</code></a></p></td>
<td><p>Compute the number of unique values in each group</p></td>
</tr>
<tr class="row-odd"><td><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.prod.html" title="pandas.core.groupby.DataFrameGroupBy.prod"><code>prod()</code></a></p></td>
<td><p>Compute the product of the values in each group</p></td>
</tr>
<tr class="row-even"><td><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.quantile.html" title="pandas.core.groupby.DataFrameGroupBy.quantile"><code>quantile()</code></a></p></td>
<td><p>Compute a given quantile of the values in each group</p></td>
</tr>
<tr class="row-odd"><td><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.sem.html" title="pandas.core.groupby.DataFrameGroupBy.sem"><code>sem()</code></a></p></td>
<td><p>Compute the standard error of the mean of the values in each group</p></td>
</tr>
<tr class="row-even"><td><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.size.html" title="pandas.core.groupby.DataFrameGroupBy.size"><code>size()</code></a></p></td>
<td><p>Compute the number of values in each group</p></td>
</tr>
<tr class="row-odd"><td><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.skew.html" title="pandas.core.groupby.DataFrameGroupBy.skew"><code>skew()</code></a> *</p></td>
<td><p>Compute the skew of the values in each group</p></td>
</tr>
<tr class="row-even"><td><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.std.html" title="pandas.core.groupby.DataFrameGroupBy.std"><code>std()</code></a></p></td>
<td><p>Compute the standard deviation of the values in each group</p></td>
</tr>
<tr class="row-odd"><td><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.sum.html" title="pandas.core.groupby.DataFrameGroupBy.sum"><code>sum()</code></a></p></td>
<td><p>Compute the sum of the values in each group</p></td>
</tr>
<tr class="row-even"><td><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.var.html" title="pandas.core.groupby.DataFrameGroupBy.var"><code>var()</code></a></p></td>
<td><p>Compute the variance of the values in each group</p></td>
</tr>
</tbody>
</table>

<p>Some examples:</p>

In [86]:
df.groupby("A")[["C", "D"]].max()

,C,D
A,,
bar,0.569972,0.725827
foo,0.905717,0.615694


In [87]:
df.groupby(["A", "B"]).mean()

C         D
A   B                        
bar one   -0.539168 -0.465201
    three  0.569972  0.725827
    two   -1.498925  0.274429
foo one    0.707187 -0.437591
    three  0.173850  0.056050
    two   -1.006240 -0.125610

<p>Another aggregation example is to compute the size of each group. This is included in GroupBy as the <code>size</code> method. It returns a Series whose index consists of the group names and the values are the sizes of each group.</p>

In [88]:
grouped = df.groupby(["A", "B"])

In [89]:
grouped.size()

A    B    
bar  one      1
     three    1
     two      1
foo  one      2
     three    1
     two      2
dtype: int64

<p>While the <a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.describe.html" title="pandas.core.groupby.DataFrameGroupBy.describe"><code>DataFrameGroupBy.describe()</code></a> method is not itself a reducer, it can be used to conveniently produce a collection of summary statistics about each of the groups.</p>

In [90]:
grouped.describe()

C                                                              \
          count      mean       std       min       25%       50%       75%   
A   B                                                                         
bar one     1.0 -0.539168       NaN -0.539168 -0.539168 -0.539168 -0.539168   
    three   1.0  0.569972       NaN  0.569972  0.569972  0.569972  0.569972   
    two     1.0 -1.498925       NaN -1.498925 -1.498925 -1.498925 -1.498925   
foo one     2.0  0.707187  0.280763  0.508658  0.607923  0.707187  0.806452   
    three   1.0  0.173850       NaN  0.173850  0.173850  0.173850  0.173850   
    two     2.0 -1.006240  1.015616 -1.724389 -1.365314 -1.006240 -0.647166   

                        D                                                    \
                max count      mean       std       min       25%       50%   
A   B                                                                         
bar one   -0.539168   1.0 -0.465201       NaN -0.465201 -0.465201 -0.465201   
    three  0.569972   1.0  0.725827       NaN  0.725827  0.725827  0.725827   
    two   -1.498925   1.0  0.274429       NaN  0.274429  0.274429  0.274429   
foo one    0.905717   2.0 -0.437591  1.226289 -1.304709 -0.871150 -0.437591   
    three  0.173850   1.0  0.056050       NaN  0.056050  0.056050  0.056050   
    two   -0.288091   2.0 -0.125610  1.048361 -0.866913 -0.496261 -0.125610   

                               
                75%       max  
A   B                          
bar one   -0.465201 -0.465201  
    three  0.725827  0.725827  
    two    0.274429  0.274429  
foo one   -0.004032  0.429526  
    three  0.056050  0.056050  
    two    0.245042  0.615694

<p>Another aggregation example is to compute the number of unique values of each group. This is similar to the <a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.value_counts.html" title="pandas.core.groupby.DataFrameGroupBy.value_counts"><code>DataFrameGroupBy.value_counts()</code></a> function, except that it only counts the number of unique values.</p>

In [91]:
ll = [['foo', 1], ['foo', 2], ['foo', 2], ['bar', 1], ['bar', 1]]

In [92]:
df4 = pd.DataFrame(ll, columns=["A", "B"])

In [93]:
df4

,A,B
0,foo,1
1,foo,2
2,foo,2
3,bar,1
4,bar,1


In [94]:
df4.groupby("A")["B"].nunique()

A
bar    1
foo    2
Name: B, dtype: int64

<div class="alert alert-block alert-info">
<p>Note</p>
<p>Aggregation functions <strong>will not</strong> return the groups that you are aggregating over as named <em>columns</em> when <code>as_index=True</code>, the default. The grouped columns will be the <strong>indices</strong> of the returned object.</p>

<p>Passing <code>as_index=False</code> <strong>will</strong> return the groups that you are aggregating over as named columns, regardless if they are named <strong>indices</strong> or <em>columns</em> in the inputs.</p>
</div>

## <h3>The <a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.aggregate.html" title="pandas.core.groupby.DataFrameGroupBy.aggregate"><code>aggregate()</code></a> method</h3>

<div class="alert alert-block alert-info">
<p>Note</p>
<p>The <a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.aggregate.html" title="pandas.core.groupby.DataFrameGroupBy.aggregate"><code>aggregate()</code></a> method can accept many different types of
inputs. This section details using string aliases for various GroupBy methods; other inputs are detailed in the sections below.</p>
</div>

<p>Any reduction method that pandas implements can be passed as a string to <a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.aggregate.html" title="pandas.core.groupby.DataFrameGroupBy.aggregate"><code>aggregate()</code></a>. Users are encouraged to use the shorthand,
<code>agg</code>. It will operate as if the corresponding method was called.</p>

In [95]:
grouped = df.groupby("A")

In [96]:
grouped[["C", "D"]].aggregate("sum")

,C,D
A,,
bar,-1.468122,0.535055
foo,-0.424255,-1.070352


In [97]:
grouped = df.groupby(["A", "B"])

In [98]:
grouped.agg("sum")

C         D
A   B                        
bar one   -0.539168 -0.465201
    three  0.569972  0.725827
    two   -1.498925  0.274429
foo one    1.414375 -0.875182
    three  0.173850  0.056050
    two   -2.012480 -0.251219

<p>The result of the aggregation will have the group names as the new index. In the case of multiple keys, the result is a <a href="https://pandas.pydata.org/docs/user_guide/advanced.html#advanced-hierarchical">MultiIndex</a> by default.
As mentioned above, this can be changed by using the <code>as_index</code> option:</p>

In [99]:
grouped = df.groupby(["A", "B"], as_index=False)

In [100]:
grouped.agg("sum")

,A,B,C,D
0,bar,one,-0.539168,-0.465201
1,bar,three,0.569972,0.725827
2,bar,two,-1.498925,0.274429
3,foo,one,1.414375,-0.875182
4,foo,three,0.173850,0.056050
5,foo,two,-2.012480,-0.251219


In [101]:
df.groupby("A", as_index=False)[["C", "D"]].agg("sum")

,A,C,D
0,bar,-1.468122,0.535055
1,foo,-0.424255,-1.070352


<p>Note that you could use the <a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reset_index.html" title="pandas.DataFrame.reset_index"><code>DataFrame.reset_index()</code></a> DataFrame function to achieve the same result as the column names are stored in the resulting <code>MultiIndex</code>, although this will make an extra copy.</p>

In [102]:
df.groupby(["A", "B"]).agg("sum").reset_index()

,A,B,C,D
0,bar,one,-0.539168,-0.465201
1,bar,three,0.569972,0.725827
2,bar,two,-1.498925,0.274429
3,foo,one,1.414375,-0.875182
4,foo,three,0.173850,0.056050
5,foo,two,-2.012480,-0.251219


## <h3>Aggregation with User-Defined Functions</h3>

<p>Users can also provide their own User-Defined Functions (UDFs) for custom aggregations.</p>

<div class="alert alert-block alert-warning">
<p>Warning</p>
<p>When aggregating with a UDF, the UDF should not mutate the provided <code>Series</code>. See <a href="https://pandas.pydata.org/docs/user_guide/gotchas.html#gotchas-udf-mutation">Mutating with User Defined Function (UDF) methods</a> for more information.</p>
</div>

<div class="alert alert-block alert-info">
<p>Note</p>
<p>Aggregating with a UDF is often less performant than using the pandas built-in methods on GroupBy. Consider breaking up a complex operation into a chain of operations that utilize the built-in methods.</p>
</div>

In [103]:
animals

,kind,height,weight
0,cat,9.1,7.9
1,dog,6.0,7.5
2,cat,9.5,9.9
3,dog,34.0,198.0


In [104]:
animals.groupby("kind")[["height"]].agg(lambda x: set(x))

,height
kind,
cat,"{9.1, 9.5}"
dog,"{34.0, 6.0}"


<p>The resulting dtype will reflect that of the aggregating function. If the results from different groups have different dtypes, then a common dtype will be determined in the same way as <code>DataFrame</code> construction.</p>

In [105]:
animals.groupby("kind")[["height"]].agg(lambda x: x.astype(int).sum())

,height
kind,
cat,18
dog,40


## <h3>Applying multiple functions at once</h3>

<p>On a grouped <code>Series</code>, you can pass a list or dict of functions to <code>SeriesGroupBy.agg()</code>, outputting a DataFrame:</p>

In [106]:
grouped = df.groupby("A")

In [107]:
grouped["C"].agg(["sum", "mean", "std"])

,sum,mean,std
A,,,
bar,-1.468122,-0.489374,1.035347
foo,-0.424255,-0.084851,1.016095


<p>On a grouped <code>DataFrame</code>, you can pass a list of functions to <code>DataFrameGroupBy.agg()</code> to aggregate each column, which produces an aggregated result with a hierarchical column index:</p>

In [108]:
grouped[["C", "D"]].agg(["sum", "mean", "std"])

C                             D                    
          sum      mean       std       sum      mean       std
A                                                              
bar -1.468122 -0.489374  1.035347  0.535055  0.178352  0.601299
foo -0.424255 -0.084851  1.016095 -1.070352 -0.214070  0.835372

<p>The resulting aggregations are named after the functions themselves. If you need to rename, then you can add in a chained operation for a <code>Series</code> like this:</p>

In [109]:
(
    grouped["C"]
    .agg(["sum", "mean", "std"])
    .rename(columns={"sum": "foo", "mean": "bar", "std": "baz"})
)

,foo,bar,baz
A,,,
bar,-1.468122,-0.489374,1.035347
foo,-0.424255,-0.084851,1.016095


<p>For a grouped <code>DataFrame</code>, you can rename in a similar manner:</p>

In [110]:
(
    grouped[["C", "D"]].agg(["sum", "mean", "std"]).rename(
        columns={"sum": "foo", "mean": "bar", "std": "baz"}
    )
)

C                             D                    
          foo       bar       baz       foo       bar       baz
A                                                              
bar -1.468122 -0.489374  1.035347  0.535055  0.178352  0.601299
foo -0.424255 -0.084851  1.016095 -1.070352 -0.214070  0.835372

<div class="alert alert-block alert-info">
<p>Note</p>
<p>In general, the output column names should be unique, but pandas will allow you apply to the same function (or two functions with the same name) to the same
column.</p>

In [111]:
grouped["C"].agg(["sum", "sum"])

,sum,sum
A,,
bar,-1.468122,-1.468122
foo,-0.424255,-0.424255


<p>pandas also allows you to provide multiple lambdas. In this case, pandas will mangle the name of the (nameless) lambda functions, appending <code>_&lt;i&gt;</code> to each subsequent lambda.</p>

In [112]:
grouped["C"].agg([lambda x: x.max() - x.min(), lambda x: x.median() - x.mean()])

,<lambda_0>,<lambda_1>
A,,
bar,2.068896,-0.049794
foo,2.630105,0.258701


## <h3>Named aggregation</h3>

<p>To support column-specific aggregation <em>with control over the output column names</em>, pandas accepts the special syntax in <a href="https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.core.groupby.DataFrameGroupBy.agg.html" title="pandas.core.groupby.DataFrameGroupBy.agg"><code>DataFrameGroupBy.agg()</code></a> and <a href="https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.core.groupby.SeriesGroupBy.agg.html" title="pandas.core.groupby.SeriesGroupBy.agg"><code>SeriesGroupBy.agg()</code></a>, known as “named aggregation”, where</p>

<ul class="simple">
<li><p>The keywords are the <em>output</em> column names</p></li>
<li><p>The values are tuples whose first element is the column to select and the second element is the aggregation to apply to that column. pandas provides the <a href="https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.NamedAgg.html" title="pandas.NamedAgg"><code>NamedAgg</code></a> namedtuple with the fields <code>['column', 'aggfunc']</code> to make it clearer what the arguments are. As usual, the aggregation can be a callable or a string alias.</p></li>
</ul>

In [113]:
animals

,kind,height,weight
0,cat,9.1,7.9
1,dog,6.0,7.5
2,cat,9.5,9.9
3,dog,34.0,198.0


In [114]:
animals.groupby("kind").agg(
    min_height=pd.NamedAgg(column="height", aggfunc="min"),
    max_height=pd.NamedAgg(column="height", aggfunc="max"),
    average_weight=pd.NamedAgg(column="weight", aggfunc="mean"),
)

,min_height,max_height,average_weight
kind,,,
cat,9.1,9.5,8.90
dog,6.0,34.0,102.75


<p><a href="https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.NamedAgg.html" title="pandas.NamedAgg"><code>NamedAgg</code></a> is just a <code>namedtuple</code>. Plain tuples are allowed as well.</p>

In [115]:
animals.groupby("kind").agg(
    min_height=("height", "min"),
    max_height=("height", "max"),
    average_weight=("weight", "mean"),
)

,min_height,max_height,average_weight
kind,,,
cat,9.1,9.5,8.90
dog,6.0,34.0,102.75


<p>If the column names you want are not valid Python keywords, construct a dictionary and unpack the keyword arguments</p>

In [116]:
animals.groupby("kind").agg(
    **{
        "total weight": pd.NamedAgg(column="weight", aggfunc="sum")
    }
)

,total weight
kind,
cat,17.8
dog,205.5


<p>When using named aggregation, additional keyword arguments are not passed through to the aggregation functions; only pairs of <code>(column, aggfunc)</code> should be passed as <code>**kwargs</code>. If your aggregation functions require additional arguments, apply them partially with <code>functools.partial()</code>.</p>

<p>Named aggregation is also valid for Series groupby aggregations. In this case there’s no column selection, so the values are just the functions.</p>

In [117]:
animals.groupby("kind").height.agg(
    min_height="min",
    max_height="max"
)

,min_height,max_height
kind,,
cat,9.1,9.5
dog,6.0,34.0


## <h3>Applying different functions to DataFrame columns</h3>

<p>By passing a dict to <code>aggregate</code> you can apply a different aggregation to the columns of a DataFrame:</p>

In [118]:
grouped.agg({"C": "sum", "D": lambda x: np.std(x, ddof=1)})

,C,D
A,,
bar,-1.468122,0.601299
foo,-0.424255,0.835372


<p>The function names can also be strings. In order for a string to be valid it must be implemented on GroupBy:</p>

In [119]:
grouped.agg({"C": "sum", "D": "std"})

,C,D
A,,
bar,-1.468122,0.601299
foo,-0.424255,0.835372


# <h2>Transformation</h2>

<p>A transformation is a GroupBy operation whose result is indexed the same as the one being grouped. Common examples include <a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.cumsum.html" title="pandas.core.groupby.DataFrameGroupBy.cumsum"><code>cumsum()</code></a> and
<a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.diff.html" title="pandas.core.groupby.DataFrameGroupBy.diff"><code>diff()</code></a>.</p>

In [120]:
speeds

,class,order,max_speed
falcon,bird,Falconiformes,389.0
parrot,bird,Psittaciformes,24.0
lion,mammal,Carnivora,80.2
monkey,mammal,Primates,NaN
leopard,mammal,Carnivora,58.0


In [121]:
grouped = speeds.groupby("class")["max_speed"]

In [122]:
grouped.cumsum()

falcon     389.0
parrot     413.0
lion        80.2
monkey       NaN
leopard    138.2
Name: max_speed, dtype: float64

In [123]:
grouped.diff()

falcon       NaN
parrot    -365.0
lion         NaN
monkey       NaN
leopard      NaN
Name: max_speed, dtype: float64

<p>Unlike aggregations, the groupings that are used to split the original object are not included in the result.</p>

<div class="alert alert-block alert-info">
<p>Note</p>
<p>Since transformations do not include the groupings that are used to split the result, the arguments <code>as_index</code> and <code>sort</code> in <a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html" title="pandas.DataFrame.groupby"><code>DataFrame.groupby()</code></a> and
<a href="https://pandas.pydata.org/docs/reference/api/pandas.Series.groupby.html" title="pandas.Series.groupby"><code>Series.groupby()</code></a> have no effect.</p>
</div>

<p>A common use of a transformation is to add the result back into the original DataFrame.</p>

In [124]:
result = speeds.copy()

In [125]:
result["cumsum"] = grouped.cumsum()

In [126]:
result["diff"] = grouped.diff()

In [127]:
result

,class,order,max_speed,cumsum,diff
falcon,bird,Falconiformes,389.0,389.0,NaN
parrot,bird,Psittaciformes,24.0,413.0,-365.0
lion,mammal,Carnivora,80.2,80.2,NaN
monkey,mammal,Primates,NaN,NaN,NaN
leopard,mammal,Carnivora,58.0,138.2,NaN


## <h3>Built-in transformation methods</h3>

<p>The following methods on GroupBy act as transformations.</p>

<table class="table">
<colgroup>
<col style="width: 20.0%">
<col style="width: 80.0%">
</colgroup>
<thead>
<tr class="row-odd"><th class="head"><p>Method</p></th>
<th class="head"><p>Description</p></th>
</tr>
</thead>
<tbody>
<tr class="row-even"><td><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.bfill.html" title="pandas.core.groupby.DataFrameGroupBy.bfill"><code>bfill()</code></a></p></td>
<td><p>Back fill NA values within each group</p></td>
</tr>
<tr class="row-odd"><td><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.cumcount.html" title="pandas.core.groupby.DataFrameGroupBy.cumcount"><code>cumcount()</code></a></p></td>
<td><p>Compute the cumulative count within each group</p></td>
</tr>
<tr class="row-even"><td><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.cummax.html" title="pandas.core.groupby.DataFrameGroupBy.cummax"><code>cummax()</code></a></p></td>
<td><p>Compute the cumulative max within each group</p></td>
</tr>
<tr class="row-odd"><td><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.cummin.html" title="pandas.core.groupby.DataFrameGroupBy.cummin"><code>cummin()</code></a></p></td>
<td><p>Compute the cumulative min within each group</p></td>
</tr>
<tr class="row-even"><td><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.cumprod.html" title="pandas.core.groupby.DataFrameGroupBy.cumprod"><code>cumprod()</code></a></p></td>
<td><p>Compute the cumulative product within each group</p></td>
</tr>
<tr class="row-odd"><td><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.cumsum.html" title="pandas.core.groupby.DataFrameGroupBy.cumsum"><code>cumsum()</code></a></p></td>
<td><p>Compute the cumulative sum within each group</p></td>
</tr>
<tr class="row-even"><td><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.diff.html" title="pandas.core.groupby.DataFrameGroupBy.diff"><code>diff()</code></a></p></td>
<td><p>Compute the difference between adjacent values within each group</p></td>
</tr>
<tr class="row-odd"><td><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.ffill.html" title="pandas.core.groupby.DataFrameGroupBy.ffill"><code>ffill()</code></a></p></td>
<td><p>Forward fill NA values within each group</p></td>
</tr>
<tr class="row-even"><td><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.pct_change.html" title="pandas.core.groupby.DataFrameGroupBy.pct_change"><code>pct_change()</code></a></p></td>
<td><p>Compute the percent change between adjacent values within each group</p></td>
</tr>
<tr class="row-odd"><td><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.rank.html" title="pandas.core.groupby.DataFrameGroupBy.rank"><code>rank()</code></a></p></td>
<td><p>Compute the rank of each value within each group</p></td>
</tr>
<tr class="row-even"><td><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.shift.html" title="pandas.core.groupby.DataFrameGroupBy.shift"><code>shift()</code></a></p></td>
<td><p>Shift values up or down within each group</p></td>
</tr>
</tbody>
</table>

<p>In addition, passing any built-in aggregation method as a string to
<a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.transform.html" title="pandas.core.groupby.DataFrameGroupBy.transform"><code>transform()</code></a> (see the next section) will broadcast the result across the group, producing a transformed result. If the aggregation method has an efficient implementation, this will be performant as well.</p>

## <h3>The <a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.transform.html" title="pandas.core.groupby.DataFrameGroupBy.transform"><code>transform()</code></a> method</h3>

<p>Similar to the <a href="https://pandas.pydata.org/docs/user_guide/groupby.html#groupby-aggregate-agg">aggregation method</a>, the <a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.transform.html" title="pandas.core.groupby.DataFrameGroupBy.transform"><code>transform()</code></a> method can accept string aliases to the built-in transformation methods in the previous section. It can <em>also</em> accept string aliases to the built-in aggregation methods. When an aggregation method is provided, the result will be broadcast across the group.</p>

In [128]:
speeds

,class,order,max_speed
falcon,bird,Falconiformes,389.0
parrot,bird,Psittaciformes,24.0
lion,mammal,Carnivora,80.2
monkey,mammal,Primates,NaN
leopard,mammal,Carnivora,58.0


In [129]:
grouped = speeds.groupby("class")["max_speed"]

In [130]:
grouped

In [131]:
grouped.transform("cumsum")

falcon     389.0
parrot     413.0
lion        80.2
monkey       NaN
leopard    138.2
Name: max_speed, dtype: float64

In [132]:
grouped.transform("sum")

falcon     413.0
parrot     413.0
lion       138.2
monkey     138.2
leopard    138.2
Name: max_speed, dtype: float64

<p>In addition to string aliases, the <a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.transform.html" title="pandas.core.groupby.DataFrameGroupBy.transform"><code>transform()</code></a> method can also accept User-Defined Functions (UDFs). The UDF must:</p>

<ul class="simple">
<li><p>Return a result that is either the same size as the group chunk or broadcastable to the size of the group chunk (e.g., a scalar, <code>grouped.transform(lambda x: x.iloc[-1])</code>).</p></li>
<li><p>Operate column-by-column on the group chunk. The transform is applied to the first group chunk using chunk.apply.</p></li>
<li><p>Not perform in-place operations on the group chunk. Group chunks should be treated as immutable, and changes to a group chunk may produce unexpected results. See <a href="https://pandas.pydata.org/docs/user_guide/gotchas.html#gotchas-udf-mutation">Mutating with User Defined Function (UDF) methods</a> for more information.</p></li>
<li><p>(Optionally) operates on all columns of the entire group chunk at once. If this is supported, a fast path is used starting from the <em>second</em> chunk.</p></li>
</ul>

<div class="alert alert-block alert-info">
<p>Note</p>
<p>Transforming by supplying <code>transform</code> with a UDF is often less performant than using the built-in methods on GroupBy.
Consider breaking up a complex operation into a chain of operations that utilize the built-in methods.</p>
<p>All of the examples in this section can be made more performant by calling built-in methods instead of using UDFs.
See <a href="https://pandas.pydata.org/docs/user_guide/groupby.html#groupby-efficient-transforms">below for examples</a>.</p>
</div>

<div class="alert alert-block alert-warning">
<p>Changed in version 2.0.0: When using <code>.transform</code> on a grouped DataFrame and the transformation function returns a DataFrame, pandas now aligns the result’s index with the input’s index. You can call <code>.to_numpy()</code> within the transformation function to avoid alignment.</p>
</div>

<p>Similar to <a href="https://pandas.pydata.org/docs/user_guide/groupby.html#groupby-aggregate-agg">The aggregate() method</a>, the resulting dtype will reflect that of the transformation function. If the results from different groups have different dtypes, then a common dtype will be determined in the same way as <code>DataFrame</code> construction.</p>

<p>Suppose we wish to standardize the data within each group:</p>

In [133]:
index = pd.date_range("10/1/1999", periods=1100)

In [134]:
ts = pd.Series(np.random.normal(0.5, 2, 1100), index)

In [135]:
ts = ts.rolling(window=100, min_periods=100).mean().dropna()

In [136]:
ts.head()

2000-01-08    0.114204
2000-01-09    0.162605
2000-01-10    0.170012
2000-01-11    0.211044
2000-01-12    0.167689
Freq: D, dtype: float64

In [137]:
ts.tail()

2002-09-30    0.596399
2002-10-01    0.607413
2002-10-02    0.603208
2002-10-03    0.626016
2002-10-04    0.621887
Freq: D, dtype: float64

In [138]:
transformed = ts.groupby(lambda x: x.year).transform(
    lambda x: (x - x.mean()) / x.std()
)

<p>We would expect the result to now have mean 0 and standard deviation 1 within each group (up to floating-point error), which we can easily check:</p>

In [139]:
# Original Data
grouped = ts.groupby(lambda x: x.year)

In [140]:
grouped.mean()

2000    0.315604
2001    0.425096
2002    0.280377
dtype: float64

In [141]:
grouped.std()

2000    0.121565
2001    0.145225
2002    0.198349
dtype: float64

In [142]:
# Transformed Data
grouped_trans = transformed.groupby(lambda x: x.year)

In [143]:
grouped_trans.mean()

2000   -5.411951e-17
2001   -4.411235e-16
2002    6.332680e-17
dtype: float64

In [144]:
grouped_trans.std()

2000    1.0
2001    1.0
2002    1.0
dtype: float64

<p>We can also visually compare the original and transformed data sets.</p>

In [145]:
compare = pd.DataFrame({"Original": ts, "Transformed": transformed})

In [ ]:
compare.plot()

<p>Transformation functions that have lower dimension outputs are broadcast to match the shape of the input array.</p>

In [147]:
ts.groupby(lambda x: x.year).transform(lambda x: x.max() - x.min())

2000-01-08    0.627893
2000-01-09    0.627893
2000-01-10    0.627893
2000-01-11    0.627893
2000-01-12    0.627893
                ...   
2002-09-30    0.741895
2002-10-01    0.741895
2002-10-02    0.741895
2002-10-03    0.741895
2002-10-04    0.741895
Freq: D, Length: 1001, dtype: float64

<p>Another common data transform is to replace missing data with the group mean.</p>

In [148]:
cols = ["A", "B", "C"]

In [149]:
values = np.random.randn(1000, 3)

In [150]:
values[np.random.randint(0, 1000, 100), 0] = np.nan

In [151]:
values[np.random.randint(0, 1000, 50), 1] = np.nan

In [152]:
values[np.random.randint(0, 1000, 200), 2] = np.nan

In [153]:
data_df = pd.DataFrame(values, columns=cols)

In [154]:
data_df

,A,B,C
0,0.545451,NaN,-0.383857
1,-2.085257,-0.860279,NaN
2,-2.092965,-1.777444,0.678063
3,0.192519,-0.759645,NaN
4,0.419903,-1.117960,1.454445
...,...,...,...
995,-0.643136,1.053897,0.465547
996,-0.362277,0.693117,-0.868387
997,0.798326,-0.276811,-0.953625
998,-0.780599,-0.073278,0.659773


In [155]:
countries = np.array(["US", "UK", "GR", "JP"])

In [156]:
key = countries[np.random.randint(0, 4, 1000)]

In [157]:
grouped = data_df.groupby(key)

In [158]:
# Non-NA count in each group
grouped.count()

,A,B,C
GR,246,257,219
JP,236,247,215
UK,222,229,197
US,203,217,185


In [159]:
transformed = grouped.transform(lambda x: x.fillna(x.mean()))

<p>We can verify that the group means have not changed in the transformed data, and that the transformed data contains no NAs.</p>

In [160]:
grouped_trans = transformed.groupby(key)

In [161]:
grouped.mean()  # original group means

,A,B,C
GR,0.029510,-0.059778,0.064416
JP,0.038011,0.062732,0.041531
UK,-0.097386,-0.121180,0.005908
US,0.104954,0.169579,-0.019601


In [162]:
grouped_trans.mean()  # transformation did not change group means

,A,B,C
GR,0.029510,-0.059778,0.064416
JP,0.038011,0.062732,0.041531
UK,-0.097386,-0.121180,0.005908
US,0.104954,0.169579,-0.019601


In [163]:
grouped.count()  # original has some missing data points

,A,B,C
GR,246,257,219
JP,236,247,215
UK,222,229,197
US,203,217,185


In [164]:
grouped_trans.count()  # counts after transformation

,A,B,C
GR,273,273,273
JP,258,258,258
UK,242,242,242
US,227,227,227


In [165]:
grouped_trans.size()  # Verify non-NA count equals group size

GR    273
JP    258
UK    242
US    227
dtype: int64

<p id="groupby-efficient-transforms">As mentioned in the note above, each of the examples in this section can be computed more efficiently using built-in methods. In the code below, the inefficient way using a UDF is commented out and the faster alternative appears below.</p>

In [166]:
# result = ts.groupby(lambda x: x.year).transform(
#     lambda x: (x - x.mean()) / x.std()
# )
grouped = ts.groupby(lambda x: x.year)

In [167]:
result = (ts - grouped.transform("mean")) / grouped.transform("std")

In [168]:
# result = ts.groupby(lambda x: x.year).transform(lambda x: x.max() - x.min())
grouped = ts.groupby(lambda x: x.year)

In [169]:
result = grouped.transform("max") - grouped.transform("min")

In [170]:
# grouped = data_df.groupby(key)
# result = grouped.transform(lambda x: x.fillna(x.mean()))
grouped = data_df.groupby(key)

In [171]:
result = data_df.fillna(grouped.transform("mean"))

## <h3>Window and resample operations</h3>

<p>It is possible to use <code>resample()</code>, <code>expanding()</code> and <code>rolling()</code> as methods on groupbys.</p>

<p>The example below will apply the <code>rolling()</code> method on the samples of the column B, based on the groups of column A.</p>

In [172]:
df_re = pd.DataFrame({"A": [1] * 10 + [5] * 10, "B": np.arange(20)})

In [173]:
df_re

,A,B
0,1,0
1,1,1
2,1,2
3,1,3
4,1,4
5,1,5
6,1,6
7,1,7
8,1,8
9,1,9


In [174]:
df_re.groupby("A").rolling(4).B.mean()

A    
1  0      NaN
   1      NaN
   2      NaN
   3      1.5
   4      2.5
   5      3.5
   6      4.5
   7      5.5
   8      6.5
   9      7.5
5  10     NaN
   11     NaN
   12     NaN
   13    11.5
   14    12.5
   15    13.5
   16    14.5
   17    15.5
   18    16.5
   19    17.5
Name: B, dtype: float64

<p>The <code>expanding()</code> method will accumulate a given operation (<code>sum()</code> in the example) for all the members of each particular group.</p>

In [175]:
df_re.groupby("A").expanding().sum()

B
A          
1 0     0.0
  1     1.0
  2     3.0
  3     6.0
  4    10.0
  5    15.0
  6    21.0
  7    28.0
  8    36.0
  9    45.0
5 10   10.0
  11   21.0
  12   33.0
  13   46.0
  14   60.0
  15   75.0
  16   91.0
  17  108.0
  18  126.0
  19  145.0

<p>Suppose you want to use the <code>resample()</code> method to get a daily frequency in each group of your dataframe, and wish to complete the
missing values with the <code>ffill()</code> method.</p>

In [176]:
df_re = pd.DataFrame(
    {
        "date": pd.date_range(start="2016-01-01", periods=4, freq="W"),
        "group": [1, 1, 2, 2],
        "val": [5, 6, 7, 8],
    }
).set_index("date")

In [177]:
df_re

,group,val
date,,
2016-01-03,1,5
2016-01-10,1,6
2016-01-17,2,7
2016-01-24,2,8


In [178]:
df_re.groupby("group").resample("1D").ffill()

group  val
group date                  
1     2016-01-03      1    5
      2016-01-04      1    5
      2016-01-05      1    5
      2016-01-06      1    5
      2016-01-07      1    5
      2016-01-08      1    5
      2016-01-09      1    5
      2016-01-10      1    6
2     2016-01-17      2    7
      2016-01-18      2    7
      2016-01-19      2    7
      2016-01-20      2    7
      2016-01-21      2    7
      2016-01-22      2    7
      2016-01-23      2    7
      2016-01-24      2    8

# <h2>Filtration</h2>

<p>A filtration is a GroupBy operation that subsets the original grouping object. It may either filter out entire groups, part of groups, or both. Filtrations return a filtered version of the calling object, including the grouping columns when provided.
In the following example, <code>class</code> is included in the result.</p>

In [179]:
speeds

,class,order,max_speed
falcon,bird,Falconiformes,389.0
parrot,bird,Psittaciformes,24.0
lion,mammal,Carnivora,80.2
monkey,mammal,Primates,NaN
leopard,mammal,Carnivora,58.0


In [180]:
speeds.groupby("class").nth(1)

,class,order,max_speed
parrot,bird,Psittaciformes,24.0
monkey,mammal,Primates,NaN


<div class="alert alert-block alert-info">
<p>Note</p>
<p>Unlike aggregations, filtrations do not add the group keys to the index of the result. Because of this, passing <code>as_index=False</code> or <code>sort=True</code> will not affect these methods.</p>
</div>

<p>Filtrations will respect subsetting the columns of the GroupBy object.</p>

In [181]:
speeds.groupby("class")[["order", "max_speed"]].nth(1)

,order,max_speed
parrot,Psittaciformes,24.0
monkey,Primates,NaN


## <h3>Built-in filtrations</h3>

<p>The following methods on GroupBy act as filtrations. All these methods have an efficient, GroupBy-specific, implementation.</p>

<table class="table">
<colgroup>
<col style="width: 20.0%">
<col style="width: 80.0%">
</colgroup>
<thead>
<tr class="row-odd"><th class="head"><p>Method</p></th>
<th class="head"><p>Description</p></th>
</tr>
</thead>
<tbody>
<tr class="row-even"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.head.html#pandas.core.groupby.DataFrameGroupBy.head" title="pandas.core.groupby.DataFrameGroupBy.head"><code>head()</code></a></p></td>
<td><p>Select the top row(s) of each group</p></td>
</tr>
<tr class="row-odd"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.nth.html#pandas.core.groupby.DataFrameGroupBy.nth" title="pandas.core.groupby.DataFrameGroupBy.nth"><code>nth()</code></a></p></td>
<td><p>Select the nth row(s) of each group</p></td>
</tr>
<tr class="row-even"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.tail.html#pandas.core.groupby.DataFrameGroupBy.tail" title="pandas.core.groupby.DataFrameGroupBy.tail"><code>tail()</code></a></p></td>
<td><p>Select the bottom row(s) of each group</p></td>
</tr>
</tbody>
</table>

<p>Users can also use transformations along with Boolean indexing to construct complex filtrations within groups. For example, suppose we are given groups of products and their volumes, and we wish to subset the data to only the largest products capturing no more than 90% of the total volume within each group.</p>

In [182]:
product_volumes = pd.DataFrame(
    {
        "group": list("xxxxyyy"),
        "product": list("abcdefg"),
        "volume": [10, 30, 20, 15, 40, 10, 20],
    }
)

In [183]:
product_volumes

,group,product,volume
0,x,a,10
1,x,b,30
2,x,c,20
3,x,d,15
4,y,e,40
5,y,f,10
6,y,g,20


In [184]:
# Sort by volume to select the largest products first
product_volumes = product_volumes.sort_values("volume", ascending=False)

In [185]:
grouped = product_volumes.groupby("group")["volume"]

In [186]:
cumpct = grouped.cumsum() / grouped.transform("sum")

In [187]:
cumpct

4    0.571429
1    0.400000
2    0.666667
6    0.857143
3    0.866667
0    1.000000
5    1.000000
Name: volume, dtype: float64

In [188]:
significant_products = product_volumes[cumpct <= 0.9]

In [189]:
significant_products.sort_values(["group", "product"])

,group,product,volume
1,x,b,30
2,x,c,20
3,x,d,15
4,y,e,40
6,y,g,20


## <h3>The <code>filter</code> method</h3>

<div class="alert alert-block alert-info">
<p>Note</p>
<p>Filtering by supplying <code>filter</code> with a User-Defined Function (UDF) is often less performant than using the built-in methods on GroupBy.
Consider breaking up a complex operation into a chain of operations that utilize the built-in methods.</p>
</div>

<p>The <code>filter</code> method takes a User-Defined Function (UDF) that, when applied to an entire group, returns either <code>True</code> or <code>False</code>. The result of the <code>filter</code> method is then the subset of groups for which the UDF returned <code>True</code>.</p>

<p>Suppose we want to take only elements that belong to groups with a group sum greater than 2.</p>

In [190]:
sf = pd.Series([1, 1, 2, 3, 3, 3])

In [191]:
sf.groupby(sf).filter(lambda x: x.sum() > 2)

3    3
4    3
5    3
dtype: int64

<p>Another useful operation is filtering out elements that belong to groups with only a couple members.</p>

In [192]:
dff = pd.DataFrame({"A": np.arange(8), "B": list("aabbbbcc")})

In [193]:
dff.groupby("B").filter(lambda x: len(x) > 2)

,A,B
2,2,b
3,3,b
4,4,b
5,5,b


<p>Alternatively, instead of dropping the offending groups, we can return a like-indexed objects where the groups that do not pass the filter are filled with NaNs.</p>

In [194]:
dff.groupby("B").filter(lambda x: len(x) > 2, dropna=False)

,A,B
0,NaN,NaN
1,NaN,NaN
2,2.0,b
3,3.0,b
4,4.0,b
5,5.0,b
6,NaN,NaN
7,NaN,NaN


<p>For DataFrames with multiple columns, filters should explicitly specify a column as the filter criterion.</p>

In [195]:
dff["C"] = np.arange(8)

In [196]:
dff.groupby("B").filter(lambda x: len(x["C"]) > 2)

,A,B,C
2,2,b,2
3,3,b,3
4,4,b,4
5,5,b,5


# <h2>Flexible <code>apply</code></h2>

<p>Some operations on the grouped data might not fit into the aggregation, transformation, or filtration categories. For these, you can use the <code>apply</code> function.</p>

<div class="alert alert-block alert-warning">
<p>Warning</p>
<p><code>apply</code> has to try to infer from the result whether it should act as a reducer, transformer, <em>or</em> filter, depending on exactly what is passed to it. Thus the grouped column(s) may be included in the output or not. While it tries to intelligently guess how to behave, it can sometimes guess wrong.</p>
</div>

<div class="alert alert-block alert-info">
<p>Note</p>
<p>All of the examples in this section can be more reliably, and more efficiently, computed using other pandas functionality.</p>
</div>

In [197]:
df

,A,B,C,D
0,foo,one,0.508658,0.429526
1,bar,one,-0.539168,-0.465201
2,foo,two,-0.288091,0.615694
3,bar,three,0.569972,0.725827
4,foo,two,-1.724389,-0.866913
5,bar,two,-1.498925,0.274429
6,foo,one,0.905717,-1.304709
7,foo,three,0.173850,0.056050


In [198]:
grouped = df.groupby("A")

In [199]:
# could also just call .describe()
grouped["C"].apply(lambda x: x.describe())

A         
bar  count    3.000000
     mean    -0.489374
     std      1.035347
     min     -1.498925
     25%     -1.019047
     50%     -0.539168
     75%      0.015402
     max      0.569972
foo  count    5.000000
     mean    -0.084851
     std      1.016095
     min     -1.724389
     25%     -0.288091
     50%      0.173850
     75%      0.508658
     max      0.905717
Name: C, dtype: float64

<p>The dimension of the returned result can also change:</p>

In [200]:
grouped = df.groupby('A')['C']

In [201]:
def f(group):
    return pd.DataFrame({'original': group,
                         'demeaned': group - group.mean()})

In [202]:
grouped.apply(f)

original  demeaned
A                        
bar 1 -0.539168 -0.049794
    3  0.569972  1.059345
    5 -1.498925 -1.009551
foo 0  0.508658  0.593509
    2 -0.288091 -0.203240
    4 -1.724389 -1.639538
    6  0.905717  0.990568
    7  0.173850  0.258701

<p><code>apply</code> on a Series can operate on a returned value from the applied function that is itself a series, and possibly upcast the result to a DataFrame:</p>

In [203]:
def f(x):
    return pd.Series([x, x ** 2], index=["x", "x^2"])

In [204]:
s = pd.Series(np.random.rand(5))

In [205]:
s

0    0.026706
1    0.149288
2    0.199338
3    0.248248
4    0.083057
dtype: float64

In [206]:
s.apply(f)

,x,x^2
0,0.026706,0.000713
1,0.149288,0.022287
2,0.199338,0.039736
3,0.248248,0.061627
4,0.083057,0.006899


<p>Similar to <a href="https://pandas.pydata.org/docs/user_guide/groupby.html#groupby-aggregate-agg">The aggregate() method</a>, the resulting dtype will reflect that of the apply function. If the results from different groups have different dtypes, then a common dtype will be determined in the same way as <code>DataFrame</code> construction.</p>

## <h3>Control grouped column(s) placement with <code>group_keys</code></h3>

<p>To control whether the grouped column(s) are included in the indices, you can use the argument <code>group_keys</code> which defaults to <code>True</code>. Compare</p>

In [207]:
df.groupby("A", group_keys=True).apply(lambda x: x)

/var/folders/rt/k30tjkvx04vdq1qxvtwsb47m0000gn/T/ipykernel_85300/569163253.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby("A", group_keys=True).apply(lambda x: x)


A      B         C         D
A                                    
bar 1  bar    one -0.539168 -0.465201
    3  bar  three  0.569972  0.725827
    5  bar    two -1.498925  0.274429
foo 0  foo    one  0.508658  0.429526
    2  foo    two -0.288091  0.615694
    4  foo    two -1.724389 -0.866913
    6  foo    one  0.905717 -1.304709
    7  foo  three  0.173850  0.056050

<p>with</p>

In [208]:
df.groupby("A", group_keys=False).apply(lambda x: x)

/var/folders/rt/k30tjkvx04vdq1qxvtwsb47m0000gn/T/ipykernel_85300/3949777565.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby("A", group_keys=False).apply(lambda x: x)


,A,B,C,D
0,foo,one,0.508658,0.429526
1,bar,one,-0.539168,-0.465201
2,foo,two,-0.288091,0.615694
3,bar,three,0.569972,0.725827
4,foo,two,-1.724389,-0.866913
5,bar,two,-1.498925,0.274429
6,foo,one,0.905717,-1.304709
7,foo,three,0.173850,0.056050


# <h2>Numba Accelerated Routines</h2>

<div class="versionadded">
<p><span class="versionmodified added">Added in version 1.1.</p>
</div>

<p>If <a href="https://numba.pydata.org/">Numba</a> is installed as an optional dependency, the <code>transform</code> and <code>aggregate</code> methods support <code>engine='numba'</code> and <code>engine_kwargs</code> arguments.
See <a href="https://pandas.pydata.org/docs/user_guide/enhancingperf.html#enhancingperf-numba">enhancing performance with Numba</a> for general usage of the arguments and performance considerations.</p>

<p>The function signature must start with <code>values, index</code> <strong>exactly</strong> as the data belonging to each group will be passed into <code>values</code>, and the group index will be passed into <code>index</code>.</p>

<div class="alert alert-block alert-warning>
<p>Warning</p>
<p>When using <code>engine='numba'</code>, there will be no “fall back” behavior internally. The group data and group index will be passed as NumPy arrays to the JITed user defined function, and no alternative execution attempts will be tried.</p>
</div>

# <h2>Other useful features</h2>

## <h3>Exclusion of non-numeric columns></h3>

<p>Again consider the example DataFrame we’ve been looking at:</p>

In [209]:
df

,A,B,C,D
0,foo,one,0.508658,0.429526
1,bar,one,-0.539168,-0.465201
2,foo,two,-0.288091,0.615694
3,bar,three,0.569972,0.725827
4,foo,two,-1.724389,-0.866913
5,bar,two,-1.498925,0.274429
6,foo,one,0.905717,-1.304709
7,foo,three,0.173850,0.056050


<p>Suppose we wish to compute the standard deviation grouped by the <code>A</code> column. There is a slight problem, namely that we don’t care about the data in column <code>B</code> because it is not numeric. You can avoid non-numeric columns by specifying <code>numeric_only=True</code>:</p>

In [210]:
df.groupby("A").std(numeric_only=True)

,C,D
A,,
bar,1.035347,0.601299
foo,1.016095,0.835372


<p>Note that <code>df.groupby('A').colname.std().</code> is more efficient than <code>df.groupby('A').std().colname</code>. So if the result of an aggregation function is only needed over one column (here <code>colname</code>), it may be filtered <em>before</em> applying the aggregation function.</p>

In [211]:
from decimal import Decimal

In [212]:
df_dec = pd.DataFrame(
    {
        "id": [1, 2, 1, 2],
        "int_column": [1, 2, 3, 4],
        "dec_column": [
            Decimal("0.50"),
            Decimal("0.15"),
            Decimal("0.25"),
            Decimal("0.40"),
        ],
    }
)

In [213]:
df_dec.groupby(["id"])[["dec_column"]].sum()

,dec_column
id,
1,0.75
2,0.55


## <h3>Handling of (un)observed Categorical values</h3>

<p>When using a <code>Categorical</span></code> grouper (as a single grouper, or as part of multiple groupers), the <code>observed</span></code> keyword controls whether to return a cartesian product of all possible groupers values (<code>observed=False</span></code>) or only those that are observed groupers (<code>observed=True</span></code>).</p>

<p>Show all values:</p>

In [214]:
pd.Series([1, 1, 1]).groupby(
    pd.Categorical(["a", "a", "a"], categories=["a", "b"]), observed=False
).count()

a    3
b    0
dtype: int64

<p>Show only the observed values:</p>

In [215]:
pd.Series([1, 1, 1]).groupby(
    pd.Categorical(["a", "a", "a"], categories=["a", "b"]), observed=True
).count()

a    3
dtype: int64

<p>The returned dtype of the grouped will <em>always</em> include <em>all</em> of the categories that were grouped.</p>

In [216]:
s = (pd.Series([1, 1, 1])
        .groupby(pd.Categorical(["a", "a", "a"], categories=["a", "b"]), observed=True)
        .count()
)

In [217]:
s.index.dtype

CategoricalDtype(categories=['a', 'b'], ordered=False, categories_dtype=object)

## <h3>NA group handling</h3>

<p>By <code>NA</span></code>, we are referring to any <code>NA</span></code> values, including <a href="../reference/api/pandas.NA.html#pandas.NA" title="pandas.NA"><code>NA</code></a>, <code>NaN</code>, <code>NaT</code>, and <code>None</code>. If there are any <code>NA</code> values in the grouping key, by default these will be excluded. In other words, any “<code>NA</code> group” will be dropped. You can include NA groups by specifying <code>dropna=False</code>.</p>

In [218]:
df = pd.DataFrame({"key": [1.0, 1.0, np.nan, 2.0, np.nan], "A": [1, 2, 3, 4, 5]})

In [219]:
df

,key,A
0,1.0,1
1,1.0,2
2,NaN,3
3,2.0,4
4,NaN,5


In [220]:
df.groupby("key", dropna=True).sum()

,A
key,
1.0,3
2.0,4


In [221]:
df.groupby("key", dropna=False).sum()

,A
key,
1.0,3
2.0,4
NaN,8


## <h3>Grouping with ordered factors</h3>

<p>Categorical variables represented as instances of pandas’s <code>Categorical</code> class can be used as group keys. If so, the order of the levels will be preserved. When <code>observed=False</span></code> and <code>sort=False</span></code>, any unobserved categories will be at the
end of the result in order.</p>

In [222]:
days = pd.Categorical(
        values=["Wed", "Mon", "Thu", "Mon", "Wed", "Sat"],
    categories=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"],
)

In [223]:
data = pd.DataFrame(
   {
       "day": days,
       "workers": [3, 4, 1, 4, 2, 2],
   }    
)

In [224]:
data

,day,workers
0,Wed,3
1,Mon,4
2,Thu,1
3,Mon,4
4,Wed,2
5,Sat,2


In [225]:
data.groupby("day", observed=False, sort=True).sum()

,workers
day,
Mon,8
Tue,0
Wed,5
Thu,1
Fri,0
Sat,2
Sun,0


In [226]:
data.groupby("day", observed=False, sort=False).sum()

,workers
day,
Wed,5
Mon,8
Thu,1
Sat,2
Tue,0
Fri,0
Sun,0


## <h3>Grouping with a grouper specification</h3>

<p>You may need to specify a bit more data to properly group. You can use the <code>pd.Grouper</code> to provide this local control.</p>

In [227]:
import datetime

In [228]:
df = pd.DataFrame(
    {
        "Branch": "A A A A A A A B".split(),
        "Buyer": "Carl Mark Carl Carl Joe Joe Joe Carl".split(),
        "Quantity": [1, 3, 5, 1, 8, 1, 9, 3],
        "Date": [
            datetime.datetime(2013, 1, 1, 13, 0),
            datetime.datetime(2013, 1, 1, 13, 5),
            datetime.datetime(2013, 10, 1, 20, 0),
            datetime.datetime(2013, 10, 2, 10, 0),
            datetime.datetime(2013, 10, 1, 20, 0),
            datetime.datetime(2013, 10, 2, 10, 0),
            datetime.datetime(2013, 12, 2, 12, 0),
            datetime.datetime(2013, 12, 2, 14, 0),
        ],
    }
)

In [229]:
df

,Branch,Buyer,Quantity,Date
0,A,Carl,1,2013-01-01 13:00:00
1,A,Mark,3,2013-01-01 13:05:00
2,A,Carl,5,2013-10-01 20:00:00
3,A,Carl,1,2013-10-02 10:00:00
4,A,Joe,8,2013-10-01 20:00:00
5,A,Joe,1,2013-10-02 10:00:00
6,A,Joe,9,2013-12-02 12:00:00
7,B,Carl,3,2013-12-02 14:00:00


<p>Groupby a specific column with the desired frequency. This is like resampling.</p>

In [230]:
df.groupby([pd.Grouper(freq="1ME", key="Date"), "Buyer"])[["Quantity"]].sum()

Quantity
Date       Buyer          
2013-01-31 Carl          1
           Mark          3
2013-10-31 Carl          6
           Joe           9
2013-12-31 Carl          3
           Joe           9

<p>When <code>freq</span></code> is specified, the object returned by <code>pd.Grouper</span></code> will be an instance of <code>pandas.api.typing.TimeGrouper</code>. When there is a column and index with the same name, you can use <code>key</code> to group by the column and <code>level</span></code> to group by the index.</p>

In [232]:
df = df.set_index("Date")

In [233]:
df["Date"] = df.index + pd.offsets.MonthEnd(2)

In [236]:
df.groupby([pd.Grouper(freq="6ME", key="Date"), "Buyer"])[["Quantity"]].sum()

Quantity
Date       Buyer          
2013-02-28 Carl          1
           Mark          3
2014-02-28 Carl          9
           Joe          18

In [237]:
df.groupby([pd.Grouper(freq="6ME", level="Date"), "Buyer"])[["Quantity"]].sum()

Quantity
Date       Buyer          
2013-01-31 Carl          1
           Mark          3
2014-01-31 Carl          9
           Joe          18

## <h3>Taking the first rows of each group</h3>

<p>Just like for a DataFrame or Series you can call head and tail on a groupby:</p>

In [238]:
df = pd.DataFrame([[1, 2], [1, 4], [5, 6]], columns=["A", "B"])

In [239]:
df

,A,B
0,1,2
1,1,4
2,5,6


In [241]:
g = df.groupby("A")

In [244]:
g.head(1)

,A,B
0,1,2
2,5,6


In [245]:
g.tail(1)

,A,B
1,1,4
2,5,6


<p>This shows the first or last n rows from each group.</p>

## <h3>Taking the nth row of each group</h3>

<p>To select the nth item from each group, use
<a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.nth.html" title="pandas.core.groupby.DataFrameGroupBy.nth"><code>DataFrameGroupBy.nth()</code></a> or
<a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.SeriesGroupBy.nth.html" title="pandas.core.groupby.SeriesGroupBy.nth"><code>SeriesGroupBy.nth()</code></a>. Arguments supplied can be any integer, lists of integers, slices, or lists of slices; see below for examples. When the nth element of a group does not exist an error is <em>not</em> raised; instead no corresponding rows are returned.</p>

<p>In general this operation acts as a filtration. In certain cases it will also return one row per group, making it also a reduction. However because in general it can return zero or multiple rows per group, pandas treats it as a filtration in all cases.</p>

In [250]:
df = pd.DataFrame([[1, np.nan], [1, 4], [5, 6]], columns=["A", "B"])

In [251]:
g = df.groupby("A")

In [252]:
g.nth(0)

,A,B
0,1,NaN
2,5,6.0


In [253]:
g.nth(-1)

,A,B
1,1,4.0
2,5,6.0


In [254]:
g.nth(1)

,A,B
1,1,4.0


<p>If the nth element of a group does not exist, then no corresponding row is included in the result. In particular, if the specified <code>n</span></code> is larger than any group, the result will be an empty DataFrame.</p>

In [255]:
g.nth(5)

,A,B


In [256]:
# nth(0) is the same as g.first()
g.nth(0, dropna="any")

,A,B
1,1,4.0
2,5,6.0


In [257]:
g.first()

,B
A,
1,4.0
5,6.0


In [258]:
# nth(-1) is the same as g.last
g.nth(-1, dropna="any")

,A,B
1,1,4.0
2,5,6.0


In [259]:
g.last()

,B
A,
1,4.0
5,6.0


In [260]:
g.B.nth(0, dropna="all")

1    4.0
2    6.0
Name: B, dtype: float64

<p>You can also select multiple rows from each group by specifying multiple nth values as a list of ints.</p>

In [261]:
business_dates = pd.date_range(start="4/1/2014", end="6/30/2014", freq="B")

In [263]:
df = pd.DataFrame(1, index=business_dates, columns=["a", "b"])

In [266]:
# get the first, 4th, and last date index for each month
df.groupby([df.index.year, df.index.month]).nth([0, 3, -1])

,a,b
2014-04-01,1,1
2014-04-04,1,1
2014-04-30,1,1
2014-05-01,1,1
2014-05-06,1,1
2014-05-30,1,1
2014-06-02,1,1
2014-06-05,1,1
2014-06-30,1,1


<p>You may also use slices or lists of slices.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell95"><span class="gp">In [250]: </span><span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">([</span><span class="n">df</span><span class="o">.</span><span class="n">index</span><span class="o">.</span><span class="n">year</span><span class="p">,</span> <span class="n">df</span><span class="o">.</span><span class="n">index</span><span class="o">.</span><span class="n">month</span><span class="p">])</span><span class="o">.</span><span class="n">nth</span><span class="p">[</span><span class="mi">1</span><span class="p">:]</span>
<span class="gh">Out[250]: </span>
<span class="go">            a  b</span>
<span class="go">2014-04-02  1  1</span>
<span class="go">2014-04-03  1  1</span>
<span class="go">2014-04-04  1  1</span>
<span class="go">2014-04-07  1  1</span>
<span class="go">2014-04-08  1  1</span>
<span class="go">...        .. ..</span>
<span class="go">2014-06-24  1  1</span>
<span class="go">2014-06-25  1  1</span>
<span class="go">2014-06-26  1  1</span>
<span class="go">2014-06-27  1  1</span>
<span class="go">2014-06-30  1  1</span>

<span class="go">[62 rows x 2 columns]</span>

<span class="gp">In [251]: </span><span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">([</span><span class="n">df</span><span class="o">.</span><span class="n">index</span><span class="o">.</span><span class="n">year</span><span class="p">,</span> <span class="n">df</span><span class="o">.</span><span class="n">index</span><span class="o">.</span><span class="n">month</span><span class="p">])</span><span class="o">.</span><span class="n">nth</span><span class="p">[</span><span class="mi">1</span><span class="p">:,</span> <span class="p">:</span><span class="o">-</span><span class="mi">1</span><span class="p">]</span>
<span class="gh">Out[251]: </span>
<span class="go">            a  b</span>
<span class="go">2014-04-01  1  1</span>
<span class="go">2014-04-02  1  1</span>
<span class="go">2014-04-03  1  1</span>
<span class="go">2014-04-04  1  1</span>
<span class="go">2014-04-07  1  1</span>
<span class="go">...        .. ..</span>
<span class="go">2014-06-24  1  1</span>
<span class="go">2014-06-25  1  1</span>
<span class="go">2014-06-26  1  1</span>
<span class="go">2014-06-27  1  1</span>
<span class="go">2014-06-30  1  1</span>

<span class="go">[65 rows x 2 columns]</span>
</pre>
</div>
</div>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell95"><span class="gp">In [250]: </span><span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">([</span><span class="n">df</span><span class="o">.</span><span class="n">index</span><span class="o">.</span><span class="n">year</span><span class="p">,</span> <span class="n">df</span><span class="o">.</span><span class="n">index</span><span class="o">.</span><span class="n">month</span><span class="p">])</span><span class="o">.</span><span class="n">nth</span><span class="p">[</span><span class="mi">1</span><span class="p">:]</span>
<span class="gh">Out[250]: </span>
<span class="go">            a  b</span>
<span class="go">2014-04-02  1  1</span>
<span class="go">2014-04-03  1  1</span>
<span class="go">2014-04-04  1  1</span>
<span class="go">2014-04-07  1  1</span>
<span class="go">2014-04-08  1  1</span>
<span class="go">...        .. ..</span>
<span class="go">2014-06-24  1  1</span>
<span class="go">2014-06-25  1  1</span>
<span class="go">2014-06-26  1  1</span>
<span class="go">2014-06-27  1  1</span>
<span class="go">2014-06-30  1  1</span>

<span class="go">[62 rows x 2 columns]</span>

<span class="gp">In [251]: </span><span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">([</span><span class="n">df</span><span class="o">.</span><span class="n">index</span><span class="o">.</span><span class="n">year</span><span class="p">,</span> <span class="n">df</span><span class="o">.</span><span class="n">index</span><span class="o">.</span><span class="n">month</span><span class="p">])</span><span class="o">.</span><span class="n">nth</span><span class="p">[</span><span class="mi">1</span><span class="p">:,</span> <span class="p">:</span><span class="o">-</span><span class="mi">1</span><span class="p">]</span>
<span class="gh">Out[251]: </span>
<span class="go">            a  b</span>
<span class="go">2014-04-01  1  1</span>
<span class="go">2014-04-02  1  1</span>
<span class="go">2014-04-03  1  1</span>
<span class="go">2014-04-04  1  1</span>
<span class="go">2014-04-07  1  1</span>
<span class="go">...        .. ..</span>
<span class="go">2014-06-24  1  1</span>
<span class="go">2014-06-25  1  1</span>
<span class="go">2014-06-26  1  1</span>
<span class="go">2014-06-27  1  1</span>
<span class="go">2014-06-30  1  1</span>

<span class="go">[65 rows x 2 columns]</span>
</pre>
</div>
</div>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell96"><span class="gp">In [252]: </span><span class="n">dfg</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">(</span><span class="nb">list</span><span class="p">(</span><span class="s2">"aaabba"</span><span class="p">),</span> <span class="n">columns</span><span class="o">=</span><span class="p">[</span><span class="s2">"A"</span><span class="p">])</span>

<span class="gp">In [253]: </span><span class="n">dfg</span>
<span class="gh">Out[253]: </span>
<span class="go">   A</span>
<span class="go">0  a</span>
<span class="go">1  a</span>
<span class="go">2  a</span>
<span class="go">3  b</span>
<span class="go">4  b</span>
<span class="go">5  a</span>

<span class="gp">In [254]: </span><span class="n">dfg</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"A"</span><span class="p">)</span><span class="o">.</span><span class="n">cumcount</span><span class="p">()</span>
<span class="gh">Out[254]: </span>
<span class="go">0    0</span>
<span class="go">1    1</span>
<span class="go">2    2</span>
<span class="go">3    0</span>
<span class="go">4    1</span>
<span class="go">5    3</span>
<span class="go">dtype: int64</span>

<span class="gp">In [255]: </span><span class="n">dfg</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"A"</span><span class="p">)</span><span class="o">.</span><span class="n">cumcount</span><span class="p">(</span><span class="n">ascending</span><span class="o">=</span><span class="kc">False</span><span class="p">)</span>
<span class="gh">Out[255]: </span>
<span class="go">0    3</span>
<span class="go">1    2</span>
<span class="go">2    1</span>
<span class="go">3    1</span>
<span class="go">4    0</span>
<span class="go">5    0</span>
<span class="go">dtype: int64</span>
</pre>
</div>
</div>

## <h3>Enumerate groups</h3>

<p>To see the ordering of the groups (as opposed to the order of rows within a group given by <code>cumcount</code>) you can use
<a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.ngroup.html" title="pandas.core.groupby.DataFrameGroupBy.ngroup"><code>DataFrameGroupBy.ngroup()</code></a>.</p>

<p>Note that the numbers given to the groups match the order in which the groups would be seen when iterating over the groupby object, not the order they are first observed.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell97"><span class="gp">In [256]: </span><span class="n">dfg</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">(</span><span class="nb">list</span><span class="p">(</span><span class="s2">"aaabba"</span><span class="p">),</span> <span class="n">columns</span><span class="o">=</span><span class="p">[</span><span class="s2">"A"</span><span class="p">])</span>

<span class="gp">In [257]: </span><span class="n">dfg</span>
<span class="gh">Out[257]: </span>
<span class="go">   A</span>
<span class="go">0  a</span>
<span class="go">1  a</span>
<span class="go">2  a</span>
<span class="go">3  b</span>
<span class="go">4  b</span>
<span class="go">5  a</span>

<span class="gp">In [258]: </span><span class="n">dfg</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"A"</span><span class="p">)</span><span class="o">.</span><span class="n">ngroup</span><span class="p">()</span>
<span class="gh">Out[258]: </span>
<span class="go">0    0</span>
<span class="go">1    0</span>
<span class="go">2    0</span>
<span class="go">3    1</span>
<span class="go">4    1</span>
<span class="go">5    0</span>
<span class="go">dtype: int64</span>

<span class="gp">In [259]: </span><span class="n">dfg</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"A"</span><span class="p">)</span><span class="o">.</span><span class="n">ngroup</span><span class="p">(</span><span class="n">ascending</span><span class="o">=</span><span class="kc">False</span><span class="p">)</span>
<span class="gh">Out[259]: </span>
<span class="go">0    1</span>
<span class="go">1    1</span>
<span class="go">2    1</span>
<span class="go">3    0</span>
<span class="go">4    0</span>
<span class="go">5    1</span>
<span class="go">dtype: int64</span>
</pre>
</div>
</div>

## <h3>Plotting</h3>

<p>Groupby also works with some plotting methods.  In this case, suppose we suspect that the values in column 1 are 3 times higher on average in group “B”.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell98"><span class="gp">In [260]: </span><span class="n">np</span><span class="o">.</span><span class="n">random</span><span class="o">.</span><span class="n">seed</span><span class="p">(</span><span class="mi">1234</span><span class="p">)</span>

<span class="gp">In [261]: </span><span class="n">df</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">(</span><span class="n">np</span><span class="o">.</span><span class="n">random</span><span class="o">.</span><span class="n">randn</span><span class="p">(</span><span class="mi">50</span><span class="p">,</span> <span class="mi">2</span><span class="p">))</span>

<span class="gp">In [262]: </span><span class="n">df</span><span class="p">[</span><span class="s2">"g"</span><span class="p">]</span> <span class="o">=</span> <span class="n">np</span><span class="o">.</span><span class="n">random</span><span class="o">.</span><span class="n">choice</span><span class="p">([</span><span class="s2">"A"</span><span class="p">,</span> <span class="s2">"B"</span><span class="p">],</span> <span class="n">size</span><span class="o">=</span><span class="mi">50</span><span class="p">)</span>

<span class="gp">In [263]: </span><span class="n">df</span><span class="o">.</span><span class="n">loc</span><span class="p">[</span><span class="n">df</span><span class="p">[</span><span class="s2">"g"</span><span class="p">]</span> <span class="o">==</span> <span class="s2">"B"</span><span class="p">,</span> <span class="mi">1</span><span class="p">]</span> <span class="o">+=</span> <span class="mi">3</span>
</pre>
</div>
</div>
<p>We can easily visualize this with a boxplot:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell99"><span class="gp">In [264]: </span><span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"g"</span><span class="p">)</span><span class="o">.</span><span class="n">boxplot</span><span class="p">()</span>
<span class="gh">Out[264]: </span>
<span class="go">A         Axes(0.1,0.15;0.363636x0.75)</span>
<span class="go">B    Axes(0.536364,0.15;0.363636x0.75)</span>
<span class="go">dtype: object</span>
</pre>
</div>
</div>

<p>The result of calling <code>boxplot</code> is a dictionary whose keys are the values of our grouping column <code>g</code> (“A” and “B”). The values of the resulting dictionary can be controlled by the <code>return_type</code> keyword of <code>boxplot</code>.
See the <a href="https://pandas.pydata.org/docs/user_guide/visualization.html#visualization-box">visualization documentation</a> for more.</p>

<div class="alert alert-block alert-warning>
<p>Warning</p>
<p>For historical reasons, <code>df.groupby("g").boxplot()</code> is not equivalent to <code>df.boxplot(by="g")</code>.
See <a href="https://pandas.pydata.org/docs/user_guide/visualization.html">here</a> for an explanation.</p>
</div>

## <h3>Piping function calls</h3>

<p>Similar to the functionality provided by <code>DataFrame</code> and <code>Series</code>, functions that take <code>GroupBy</code> objects can be chained together using a <code>pipe</code> method to allow for a cleaner, more readable syntax. To read about <code>.pipe</code> in general terms, see <a href="https://pandas.pydata.org/docs/user_guide/basics.html#basics-pipe">here</a>.</p>

<p>Combining <code>.groupby</code> and <code>.pipe</code> is often useful when you need to reuse
GroupBy objects.</p>

<p>As an example, imagine having a DataFrame with columns for stores, products, revenue and quantity sold. We’d like to do a groupwise calculation of <em>prices</em> (i.e. revenue/quantity) per store and per product. We could do this in a multi-step operation, but expressing it in terms of piping can make the code more readable. First we set the data:</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell100"><span class="gp">In [265]: </span><span class="n">n</span> <span class="o">=</span> <span class="mi">1000</span>

<span class="gp">In [266]: </span><span class="n">df</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">(</span>
<span class="gp"></span>    <span class="p">{</span>
<span class="gp"></span>        <span class="s2">"Store"</span><span class="p">:</span> <span class="n">np</span><span class="o">.</span><span class="n">random</span><span class="o">.</span><span class="n">choice</span><span class="p">([</span><span class="s2">"Store_1"</span><span class="p">,</span> <span class="s2">"Store_2"</span><span class="p">],</span> <span class="n">n</span><span class="p">),</span>
<span class="gp"></span>        <span class="s2">"Product"</span><span class="p">:</span> <span class="n">np</span><span class="o">.</span><span class="n">random</span><span class="o">.</span><span class="n">choice</span><span class="p">([</span><span class="s2">"Product_1"</span><span class="p">,</span> <span class="s2">"Product_2"</span><span class="p">],</span> <span class="n">n</span><span class="p">),</span>
<span class="gp"></span>        <span class="s2">"Revenue"</span><span class="p">:</span> <span class="p">(</span><span class="n">np</span><span class="o">.</span><span class="n">random</span><span class="o">.</span><span class="n">random</span><span class="p">(</span><span class="n">n</span><span class="p">)</span> <span class="o">*</span> <span class="mi">50</span> <span class="o">+</span> <span class="mi">10</span><span class="p">)</span><span class="o">.</span><span class="n">round</span><span class="p">(</span><span class="mi">2</span><span class="p">),</span>
<span class="gp"></span>        <span class="s2">"Quantity"</span><span class="p">:</span> <span class="n">np</span><span class="o">.</span><span class="n">random</span><span class="o">.</span><span class="n">randint</span><span class="p">(</span><span class="mi">1</span><span class="p">,</span> <span class="mi">10</span><span class="p">,</span> <span class="n">size</span><span class="o">=</span><span class="n">n</span><span class="p">),</span>
<span class="gp"></span>    <span class="p">}</span>
<span class="gp"></span><span class="p">)</span>
<span class="gp"></span>

<span class="gp">In [267]: </span><span class="n">df</span><span class="o">.</span><span class="n">head</span><span class="p">(</span><span class="mi">2</span><span class="p">)</span>
<span class="gh">Out[267]: </span>
<span class="go">     Store    Product  Revenue  Quantity</span>
<span class="go">0  Store_2  Product_1    26.12         1</span>
<span class="go">1  Store_2  Product_1    28.86         1</span>
</pre>
</div>
</div>

<p>We now find the prices per store/product.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell101"><span class="gp">In [268]: </span><span class="p">(</span>
<span class="gp"></span>    <span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">([</span><span class="s2">"Store"</span><span class="p">,</span> <span class="s2">"Product"</span><span class="p">])</span>
<span class="gp"></span>    <span class="o">.</span><span class="n">pipe</span><span class="p">(</span><span class="k">lambda</span> <span class="n">grp</span><span class="p">:</span> <span class="n">grp</span><span class="o">.</span><span class="n">Revenue</span><span class="o">.</span><span class="n">sum</span><span class="p">()</span> <span class="o">/</span> <span class="n">grp</span><span class="o">.</span><span class="n">Quantity</span><span class="o">.</span><span class="n">sum</span><span class="p">())</span>
<span class="gp"></span>    <span class="o">.</span><span class="n">unstack</span><span class="p">()</span>
<span class="gp"></span>    <span class="o">.</span><span class="n">round</span><span class="p">(</span><span class="mi">2</span><span class="p">)</span>
<span class="gp"></span><span class="p">)</span>
<span class="gp"></span>
<span class="gh">Out[268]: </span>
<span class="go">Product  Product_1  Product_2</span>
<span class="go">Store                        </span>
<span class="go">Store_1       6.82       7.05</span>
<span class="go">Store_2       6.30       6.64</span>
</pre>
</div>
</div>

<p>Piping can also be expressive when you want to deliver a grouped object to some arbitrary function, for example:</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell102"><span class="gp">In [269]: </span><span class="k">def</span><span class="w"> </span><span class="nf">mean</span><span class="p">(</span><span class="n">groupby</span><span class="p">):</span>
<span class="gp"></span>    <span class="k">return</span> <span class="n">groupby</span><span class="o">.</span><span class="n">mean</span><span class="p">()</span>
<span class="gp"></span>

<span class="gp">In [270]: </span><span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">([</span><span class="s2">"Store"</span><span class="p">,</span> <span class="s2">"Product"</span><span class="p">])</span><span class="o">.</span><span class="n">pipe</span><span class="p">(</span><span class="n">mean</span><span class="p">)</span>
<span class="gh">Out[270]: </span>
<span class="go">                     Revenue  Quantity</span>
<span class="go">Store   Product                       </span>
<span class="go">Store_1 Product_1  34.622727  5.075758</span>
<span class="go">        Product_2  35.482815  5.029630</span>
<span class="go">Store_2 Product_1  32.972837  5.237589</span>
<span class="go">        Product_2  34.684360  5.224000</span>
</pre>
</div>
</div>

<p>Here <code>mean</span></code> takes a GroupBy object and finds the mean of the Revenue and Quantity columns respectively for each Store-Product combination. The <code>mean</span></code> function can be any function that takes in a GroupBy object; the <code>.pipe</span></code> will pass the GroupBy object as a parameter into the function you specify.</p>

# <h2>Examples</h2>

## <h3>Multi-column factorization</h3>

<p>By using <a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.ngroup.html" title="pandas.core.groupby.DataFrameGroupBy.ngroup"><code>DataFrameGroupBy.ngroup()</code></a>, we can extract information about the groups in a way similar to <a href="https://pandas.pydata.org/docs/reference/api/pandas.factorize.html" title="pandas.factorize"><code>factorize()</code></a> (as described further in the <a href="https://pandas.pydata.org/docs/user_guide/reshaping.html#reshaping-factorize">reshaping API</a>) but which applies naturally to multiple columns of mixed type and different sources.
This can be useful as an intermediate categorical-like step in processing, when the relationships between the group rows are more important than their content, or as input to an algorithm which only accepts the integer encoding. (For more information about support in pandas for full categorical data, see the
<a href="https://pandas.pydata.org/docs/user_guide/categorical.html">Categorical introduction</a> and the
<a href="https://pandas.pydata.org/docs/reference/arrays.html#api-arrays-categorical">API documentation</a>.)</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell103"><span class="gp">In [271]: </span><span class="n">dfg</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">({</span><span class="s2">"A"</span><span class="p">:</span> <span class="p">[</span><span class="mi">1</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">2</span><span class="p">,</span> <span class="mi">3</span><span class="p">,</span> <span class="mi">2</span><span class="p">],</span> <span class="s2">"B"</span><span class="p">:</span> <span class="nb">list</span><span class="p">(</span><span class="s2">"aaaba"</span><span class="p">)})</span>

<span class="gp">In [272]: </span><span class="n">dfg</span>
<span class="gh">Out[272]: </span>
<span class="go">   A  B</span>
<span class="go">0  1  a</span>
<span class="go">1  1  a</span>
<span class="go">2  2  a</span>
<span class="go">3  3  b</span>
<span class="go">4  2  a</span>

<span class="gp">In [273]: </span><span class="n">dfg</span><span class="o">.</span><span class="n">groupby</span><span class="p">([</span><span class="s2">"A"</span><span class="p">,</span> <span class="s2">"B"</span><span class="p">])</span><span class="o">.</span><span class="n">ngroup</span><span class="p">()</span>
<span class="gh">Out[273]: </span>
<span class="go">0    0</span>
<span class="go">1    0</span>
<span class="go">2    1</span>
<span class="go">3    2</span>
<span class="go">4    1</span>
<span class="go">dtype: int64</span>

<span class="gp">In [274]: </span><span class="n">dfg</span><span class="o">.</span><span class="n">groupby</span><span class="p">([</span><span class="s2">"A"</span><span class="p">,</span> <span class="p">[</span><span class="mi">0</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">1</span><span class="p">]])</span><span class="o">.</span><span class="n">ngroup</span><span class="p">()</span>
<span class="gh">Out[274]: </span>
<span class="go">0    0</span>
<span class="go">1    0</span>
<span class="go">2    1</span>
<span class="go">3    3</span>
<span class="go">4    2</span>
<span class="go">dtype: int64</span>
</pre>
</div>
</div>

## <h3>Groupby by indexer to ‘resample’ data</h3>

<p>Resampling produces new hypothetical samples (resamples) from already existing observed data or from a model that generates data. These new samples are similar to the pre-existing samples.</p>

<p>In order for resample to work on indices that are non-datetimelike, the following procedure can be utilized.</p>

<p>In the following examples, <strong>df.index // 5</strong> returns an integer array which is used to determine what gets selected for the groupby operation.</p>

<div class="alert alert-block alert-info">
<p>Note</p>
<p>The example below shows how we can downsample by consolidation of samples into fewer ones.
Here by using <strong>df.index // 5</strong>, we are aggregating the samples in bins. By applying <strong>std()</strong> function, we aggregate the information contained in many samples into a small subset of values which is their standard deviation thereby reducing the number of samples.</p>
</div>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell104"><span class="gp">In [275]: </span><span class="n">df</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">(</span><span class="n">np</span><span class="o">.</span><span class="n">random</span><span class="o">.</span><span class="n">randn</span><span class="p">(</span><span class="mi">10</span><span class="p">,</span> <span class="mi">2</span><span class="p">))</span>

<span class="gp">In [276]: </span><span class="n">df</span>
<span class="gh">Out[276]: </span>
<span class="go">          0         1</span>
<span class="go">0 -0.793893  0.321153</span>
<span class="go">1  0.342250  1.618906</span>
<span class="go">2 -0.975807  1.918201</span>
<span class="go">3 -0.810847 -1.405919</span>
<span class="go">4 -1.977759  0.461659</span>
<span class="go">5  0.730057 -1.316938</span>
<span class="go">6 -0.751328  0.528290</span>
<span class="go">7 -0.257759 -1.081009</span>
<span class="go">8  0.505895 -1.701948</span>
<span class="go">9 -1.006349  0.020208</span>

<span class="gp">In [277]: </span><span class="n">df</span><span class="o">.</span><span class="n">index</span> <span class="o">//</span> <span class="mi">5</span>
<span class="gh">Out[277]: </span><span class="go">Index([0, 0, 0, 0, 0, 1, 1, 1, 1, 1], dtype='int64')</span>

<span class="gp">In [278]: </span><span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="n">df</span><span class="o">.</span><span class="n">index</span> <span class="o">//</span> <span class="mi">5</span><span class="p">)</span><span class="o">.</span><span class="n">std</span><span class="p">()</span>
<span class="gh">Out[278]: </span>
<span class="go">          0         1</span>
<span class="go">0  0.823647  1.312912</span>
<span class="go">1  0.760109  0.942941</span>
</pre>
</div>
</div>

## <h3>Returning a Series to propagate names</h3>

<p>Group DataFrame columns, compute a set of metrics and return a named Series. The Series name is used as the name for the column index. This is especially useful in conjunction with reshaping operations such as stacking, in which the column index name will be used as the name of the inserted column:</p>


<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell105"><span class="gp">In [279]: </span><span class="n">df</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">(</span>
<span class="gp"></span>    <span class="p">{</span>
<span class="gp"></span>        <span class="s2">"a"</span><span class="p">:</span> <span class="p">[</span><span class="mi">0</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">2</span><span class="p">,</span> <span class="mi">2</span><span class="p">,</span> <span class="mi">2</span><span class="p">,</span> <span class="mi">2</span><span class="p">],</span>
<span class="gp"></span>        <span class="s2">"b"</span><span class="p">:</span> <span class="p">[</span><span class="mi">0</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">1</span><span class="p">],</span>
<span class="gp"></span>        <span class="s2">"c"</span><span class="p">:</span> <span class="p">[</span><span class="mi">1</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">0</span><span class="p">],</span>
<span class="gp"></span>        <span class="s2">"d"</span><span class="p">:</span> <span class="p">[</span><span class="mi">0</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">1</span><span class="p">],</span>
<span class="gp"></span>    <span class="p">}</span>
<span class="gp"></span><span class="p">)</span>
<span class="gp"></span>

<span class="gp">In [280]: </span><span class="k">def</span><span class="w"> </span><span class="nf">compute_metrics</span><span class="p">(</span><span class="n">x</span><span class="p">):</span>
<span class="gp"></span>    <span class="n">result</span> <span class="o">=</span> <span class="p">{</span><span class="s2">"b_sum"</span><span class="p">:</span> <span class="n">x</span><span class="p">[</span><span class="s2">"b"</span><span class="p">]</span><span class="o">.</span><span class="n">sum</span><span class="p">(),</span> <span class="s2">"c_mean"</span><span class="p">:</span> <span class="n">x</span><span class="p">[</span><span class="s2">"c"</span><span class="p">]</span><span class="o">.</span><span class="n">mean</span><span class="p">()}</span>
<span class="gp"></span>    <span class="k">return</span> <span class="n">pd</span><span class="o">.</span><span class="n">Series</span><span class="p">(</span><span class="n">result</span><span class="p">,</span> <span class="n">name</span><span class="o">=</span><span class="s2">"metrics"</span><span class="p">)</span>
<span class="gp"></span>

<span class="gp">In [281]: </span><span class="n">result</span> <span class="o">=</span> <span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"a"</span><span class="p">)</span><span class="o">.</span><span class="n">apply</span><span class="p">(</span><span class="n">compute_metrics</span><span class="p">,</span> <span class="n">include_groups</span><span class="o">=</span><span class="kc">False</span><span class="p">)</span>

<span class="gp">In [282]: </span><span class="n">result</span>
<span class="gh">Out[282]: </span>
<span class="go">metrics  b_sum  c_mean</span>
<span class="go">a                     </span>
<span class="go">0          2.0     0.5</span>
<span class="go">1          2.0     0.5</span>
<span class="go">2          2.0     0.5</span>

<span class="gp">In [283]: </span><span class="n">result</span><span class="o">.</span><span class="n">stack</span><span class="p">(</span><span class="n">future_stack</span><span class="o">=</span><span class="kc">True</span><span class="p">)</span>
<span class="gh">Out[283]: </span>
<span class="go">a  metrics</span>
<span class="go">0  b_sum      2.0</span>
<span class="go">   c_mean     0.5</span>
<span class="go">1  b_sum      2.0</span>
<span class="go">   c_mean     0.5</span>
<span class="go">2  b_sum      2.0</span>
<span class="go">   c_mean     0.5</span>
<span class="go">dtype: float64</span>
</pre>
</div>
</div>
</section>
</section>
</section>